# 07. Chicago Bears Event Sensitivity and 2024 Blind Detection  
# 07. Chicago Bears 赛事敏感度与 2024 盲测

**Pipeline position / 主线位置:** hourly pickup/dropoff signals -> known 2022-2023 events -> 2024 prediction and blind detection.  
**流程位置：** 小时级上下车信号 -> 2022-2023 已知赛事 -> 2024 预测与盲测识别。

Part A measures how pickup and dropoff change before and after known Bears home games. It uses distance-based H3 zones, robust normal baselines, and matched non-game days. Part B learns the event pattern from 2022-2023, predicts known 2024 games, then scans 2024 without the schedule and scores the frozen candidates against the real schedule.  
第一部分测量 Bears 已知主场比赛前后的上下车变化，使用按距离选择的 H3 区域、稳健普通日基线和匹配的非比赛日。第二部分从 2022-2023 学习赛事规律，先预测已知的 2024 比赛，再在不提供赛程的情况下扫描 2024，最后用真实赛程评价已经冻结的候选结果。

**Inputs / 输入:** `unified_trips_h3_res9`, official Bears schedules.  
**Outputs / 输出:** event curves, spatial sensitivity, known-schedule errors, blind Top-K metrics.

# Part A. Learn event sensitivity from 2022-2023  
# 第一部分：从 2022-2023 学习赛事敏感度

In [ ]:
# Cell 1 - Install dependencies / 安装依赖 / Run once if these packages are missing. / 如果缺少这些包，只需要运行一次。

%pip install -q pandas numpy matplotlib seaborn pymysql h3

In [ ]:
from getpass import getpass
# Cell 2 — Imports and configuration / 导入包和配置

from pathlib import Path
from IPython.display import display
import json
import math
import os
import time
import warnings

import h3
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymysql
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.unicode_minus"] = False

# Use an available Chinese font to avoid missing-glyph warnings. / 自动选择中文字体，避免图中出现方框和大量字体警告。
font_candidates = [
    "PingFang SC",
    "Arial Unicode MS",
    "Heiti TC",
    "STHeiti",
    "Songti SC",
    "Noto Sans CJK SC",
]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
CHINESE_FONT = next(
    (font for font in font_candidates if font in available_fonts),
    None,
)
if CHINESE_FONT:
    plt.rcParams["font.sans-serif"] = [CHINESE_FONT, "DejaVu Sans"]
else:
    print(
        "Chinese plot font was not found; English labels remain readable. / "
        "未找到中文绘图字体，英文标签仍可正常显示。"
    )

# Project paths / 项目路径
PROJECT_DIR = Path(
    os.getenv("CHICAGO_TNP_PROJECT_DIR", str(Path.cwd()))
).expanduser().resolve()
OUTPUT_DIR = (
    PROJECT_DIR / "notebook_outputs_bears_home_event_sensitivity_"
)
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# MatrixOne connection / MatrixOne 连接配置
MO_HOST = os.getenv("MATRIXONE_HOST", "127.0.0.1")
MO_PORT = int(os.getenv("MATRIXONE_PORT", "6001"))
MO_USER = os.getenv("MATRIXONE_USER", "root")
MO_PASSWORD = os.getenv("MATRIXONE_PASSWORD") or getpass("MatrixOne password / MatrixOne 密码: ")
MO_DATABASE = os.getenv("MATRIXONE_DATABASE", "chicago_tnp")
ANALYSIS_TABLE = os.getenv(
    "MATRIXONE_ANALYSIS_TABLE", "unified_trips_h3_res9"
)

# Soldier Field / Soldier Field 坐标
SOLDIER_FIELD_LAT = 41.862
SOLDIER_FIELD_LON = -87.616
H3_RESOLUTION = 9

# Distance-based spatial zones / 按距离定义的空间范围
MAX_SEARCH_RING_K = 8
CORE_RADIUS_KM = 0.85
EXPANDED_RADIUS_KM = 1.75
MIN_OBSERVED_TRIPS = 1
PRIMARY_ZONE = "core"

# Event timing window / 赛事时间窗口
QUERY_START = "2022-07-01"
QUERY_END = "2024-03-01"
RELATIVE_HOURS = list(range(-6, 9))
EXCLUSION_RELATIVE_HOURS = list(range(-8, 11))
ESTIMATED_GAME_DURATION_HOURS = 3.25

# Baseline settings / 基线设置
CONTROL_WINDOW_DAYS = 70
MIN_CONTROL_HOURS = 10
ROBUST_Z_THRESHOLD = 3.0

# Windows used only for summary scores. / 这些窗口只用于汇总，不代表每场比赛的实际结束时间。
ARRIVAL_SCORE_HOURS = list(range(-4, 1))
EXIT_SCORE_HOURS = list(range(3, 9))

# Cache controls / 缓存控制
FORCE_REFRESH_H3_COUNTS = False
FORCE_REFRESH_HOURLY = False

print("Project folder / 项目目录:", PROJECT_DIR)
print("Output folder / 输出目录:", OUTPUT_DIR)
print("Analysis table / 分析表:", ANALYSIS_TABLE)
print("Relative hours / 相对小时:", RELATIVE_HOURS)
print("Chinese font / 中文字体:", CHINESE_FONT)

In [ ]:
# Cell 3 - MatrixOne connection helpers / MatrixOne 连接辅助函数

def connect_matrixone():
    """
    Open a MatrixOne connection through the MySQL protocol.
    通过 MySQL 协议连接 MatrixOne。
    """
    return pymysql.connect(
        host=MO_HOST,
        port=MO_PORT,
        user=MO_USER,
        password=MO_PASSWORD,
        database=MO_DATABASE,
        charset="utf8mb4",
        connect_timeout=30,
        read_timeout=3600,
        write_timeout=3600,
        autocommit=True,
    )


def query_df(sql, params=None, retries=3):
    """
    Run SQL and return a DataFrame. Reconnect after a dropped connection.
    执行 SQL 并返回 DataFrame；连接中断时自动重连。
    """
    last_error = None
    for attempt in range(1, retries + 1):
        conn = None
        try:
            conn = connect_matrixone()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return pd.read_sql_query(sql, conn, params=params)
        except Exception as exc:
            last_error = exc
            print(
                f"Query attempt {attempt}/{retries} failed / "
                f"查询第 {attempt}/{retries} 次失败: {exc!r}"
            )
            if attempt < retries:
                time.sleep(3 * attempt)
        finally:
            if conn is not None:
                conn.close()
    raise last_error


def table_exists(table_name):
    """
    Check whether a table exists in the current database.
    检查当前数据库中是否存在指定表。
    """
    result = query_df("SHOW TABLES;")
    return table_name in set(result.iloc[:, 0].astype(str))


def sql_string(value):
    """
    Escape a Python value as a SQL string literal.
    把 Python 值安全转换成 SQL 字符串。
    """
    return "'" + str(value).replace("\\", "\\\\").replace("'", "''") + "'"


print("Connecting to MatrixOne / 正在连接 MatrixOne...")
connection_test = query_df("SELECT 1 AS connected;")
display(connection_test)

In [ ]:
# Cell 4 - Validate the full table and define the official game schedule / 验证全量表并定义官方主场赛程

if not table_exists(ANALYSIS_TABLE):
    raise RuntimeError(
        f"Missing table / 找不到数据表: {MO_DATABASE}.{ANALYSIS_TABLE}"
    )

table_check = query_df(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(trip_start_timestamp) AS min_start,
    MAX(trip_start_timestamp) AS max_start
FROM `{ANALYSIS_TABLE}`;
""")
display(table_check)

# All regular-season Bears home games at Soldier Field for the 2022 and 2023 seasons. / 2022 和 2023 赛季 Soldier Field 的全部 Bears 常规赛主场。
GAME_ROWS = [
    # 2022 season / 2022 赛季
    (2022, 1,  "San Francisco 49ers",  "2022-09-11 12:00:00"),
    (2022, 3,  "Houston Texans",       "2022-09-25 12:00:00"),
    (2022, 6,  "Washington Commanders","2022-10-13 19:15:00"),
    (2022, 9,  "Miami Dolphins",        "2022-11-06 12:00:00"),
    (2022, 10, "Detroit Lions",         "2022-11-13 12:00:00"),
    (2022, 13, "Green Bay Packers",     "2022-12-04 12:00:00"),
    (2022, 15, "Philadelphia Eagles",   "2022-12-18 12:00:00"),
    (2022, 16, "Buffalo Bills",          "2022-12-24 12:00:00"),
    (2022, 18, "Minnesota Vikings",      "2023-01-08 12:00:00"),
    # 2023 season / 2023 赛季
    (2023, 1,  "Green Bay Packers",      "2023-09-10 15:25:00"),
    (2023, 4,  "Denver Broncos",         "2023-10-01 12:00:00"),
    (2023, 6,  "Minnesota Vikings",      "2023-10-15 12:00:00"),
    (2023, 7,  "Las Vegas Raiders",      "2023-10-22 12:00:00"),
    (2023, 10, "Carolina Panthers",      "2023-11-09 19:15:00"),
    (2023, 14, "Detroit Lions",          "2023-12-10 12:00:00"),
    (2023, 16, "Arizona Cardinals",      "2023-12-24 15:25:00"),
    (2023, 17, "Atlanta Falcons",        "2023-12-31 12:00:00"),
]

games = pd.DataFrame(
    GAME_ROWS,
    columns=["season", "week", "opponent", "scheduled_kickoff_local"],
)
games["scheduled_kickoff_local"] = pd.to_datetime(
    games["scheduled_kickoff_local"]
)
games["estimated_game_end_local"] = (
    games["scheduled_kickoff_local"]
    + pd.to_timedelta(ESTIMATED_GAME_DURATION_HOURS, unit="h")
)
games["game_id"] = [
    f"{season}_W{week:02d}_{kickoff:%Y%m%d}"
    for season, week, kickoff in zip(
        games["season"],
        games["week"],
        games["scheduled_kickoff_local"],
    )
]
games["game_label"] = [
    f"{season} W{week} vs {opponent}"
    for season, week, opponent in zip(
        games["season"], games["week"], games["opponent"]
    )
]

print("Regular-season home games / 常规赛主场数量:", len(games))
display(
    games[
        [
            "game_id",
            "season",
            "week",
            "opponent",
            "scheduled_kickoff_local",
            "estimated_game_end_local",
        ]
    ]
)


# Event labels used in the analysis. / 分析使用的赛事标签。
games["kickoff_hour"] = games["scheduled_kickoff_local"].dt.floor("h")
games["is_holiday_period"] = (
    ((games["scheduled_kickoff_local"].dt.month == 12)
     & (games["scheduled_kickoff_local"].dt.day.isin([24, 25, 31])))
    | ((games["scheduled_kickoff_local"].dt.month == 1)
       & (games["scheduled_kickoff_local"].dt.day == 1))
)

print("\nHoliday-period games / 节日期间比赛:")
display(
    games.loc[
        games["is_holiday_period"],
        ["game_label", "scheduled_kickoff_local", "is_holiday_period"],
    ]
)

## A1. 构造核心区、扩展区与逐 H3 分析范围
## 1. Build core, expanded, and per-H3 analysis zones

精确 Soldier Field H3 可能没有公开 TNP centroid，因此本分析不按固定格子数量扩大 H3 ring，而是按实际距离定义两层区域：

The exact Soldier Field H3 may have no public TNP centroid, so the analysis uses distance-based zones instead of expanding a ring to reach a fixed cell count:

- `core`：H3 中心距场馆不超过 0.85 公里，作为主分析范围。  
  H3 centers within 0.85 km; this is the primary zone.
- `expanded`：H3 中心距场馆不超过 1.75 公里，只用于检查空间范围是否改变结论。  
  H3 centers within 1.75 km; used as a sensitivity check.

同时保留每个 H3 的独立结果，避免远处格子把核心场馆信号拉高。

In [ ]:
# Cell 5 - Select distance-based H3 zones / 按距离选择 H3 区域

def haversine_km(lat1, lon1, lat2, lon2):
    '''
    Calculate great-circle distance in kilometers.
    计算两个经纬度之间的球面距离（公里）。
    '''
    radius_km = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = (
        math.sin(dp / 2) ** 2
        + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    )
    return 2 * radius_km * math.asin(math.sqrt(a))


exact_stadium_h3 = h3.latlng_to_cell(
    SOLDIER_FIELD_LAT,
    SOLDIER_FIELD_LON,
    H3_RESOLUTION,
)
candidate_h3 = sorted(h3.grid_disk(exact_stadium_h3, MAX_SEARCH_RING_K))
candidate_sql = ", ".join(sql_string(cell) for cell in candidate_h3)
h3_count_cache = CACHE_DIR / "candidate_h3_counts.csv.gz"

if h3_count_cache.exists() and not FORCE_REFRESH_H3_COUNTS:
    candidate_counts = pd.read_csv(h3_count_cache)
    print("Loaded H3-count cache / 已读取 H3 计数缓存")
else:
    candidate_counts = query_df(f'''
    SELECT
        x.h3,
        SUM(x.pickup_count) AS pickup_count,
        SUM(x.dropoff_count) AS dropoff_count
    FROM (
        SELECT
            pickup_h3 AS h3,
            COUNT(*) AS pickup_count,
            0 AS dropoff_count
        FROM `{ANALYSIS_TABLE}`
        WHERE pickup_h3 IN ({candidate_sql})
          AND trip_start_timestamp >= {sql_string(QUERY_START)}
          AND trip_start_timestamp < {sql_string(QUERY_END)}
          AND COALESCE(shared_trip_authorized, 0) = 0
        GROUP BY pickup_h3

        UNION ALL

        SELECT
            dropoff_h3 AS h3,
            0 AS pickup_count,
            COUNT(*) AS dropoff_count
        FROM `{ANALYSIS_TABLE}`
        WHERE dropoff_h3 IN ({candidate_sql})
          AND trip_end_timestamp >= {sql_string(QUERY_START)}
          AND trip_end_timestamp < {sql_string(QUERY_END)}
          AND COALESCE(shared_trip_authorized, 0) = 0
        GROUP BY dropoff_h3
    ) x
    GROUP BY x.h3;
    ''')
    candidate_counts.to_csv(
        h3_count_cache, index=False, compression="gzip"
    )

candidate_counts["h3"] = candidate_counts["h3"].astype(str)
for col in ["pickup_count", "dropoff_count"]:
    candidate_counts[col] = pd.to_numeric(
        candidate_counts[col], errors="coerce"
    ).fillna(0)
candidate_counts["total_trips"] = (
    candidate_counts["pickup_count"] + candidate_counts["dropoff_count"]
)
candidate_counts = candidate_counts[
    candidate_counts["total_trips"] >= MIN_OBSERVED_TRIPS
].copy()

if candidate_counts.empty:
    raise RuntimeError(
        "No observed H3 cells near Soldier Field. / "
        "Soldier Field 附近没有找到有效 H3。"
    )

candidate_counts[["lat", "lon"]] = candidate_counts["h3"].apply(
    lambda cell: pd.Series(h3.cell_to_latlng(cell))
)
candidate_counts["distance_to_stadium_km"] = candidate_counts.apply(
    lambda row: haversine_km(
        SOLDIER_FIELD_LAT,
        SOLDIER_FIELD_LON,
        row["lat"],
        row["lon"],
    ),
    axis=1,
)
candidate_counts["grid_distance_from_exact"] = candidate_counts["h3"].apply(
    lambda cell: h3.grid_distance(exact_stadium_h3, cell)
)
candidate_counts = candidate_counts.sort_values(
    ["distance_to_stadium_km", "total_trips"],
    ascending=[True, False],
).reset_index(drop=True)

core_cells = candidate_counts.loc[
    candidate_counts["distance_to_stadium_km"] <= CORE_RADIUS_KM, "h3"
].tolist()
expanded_cells = candidate_counts.loc[
    candidate_counts["distance_to_stadium_km"] <= EXPANDED_RADIUS_KM, "h3"
].tolist()

# Always keep at least the nearest observed cell. / 至少保留离场馆最近的一个有效格子。
if not core_cells:
    core_cells = [candidate_counts.iloc[0]["h3"]]
if not expanded_cells:
    expanded_cells = list(core_cells)

ZONE_DEFINITIONS = {
    "core": sorted(set(core_cells)),
    "expanded": sorted(set(expanded_cells)),
}
all_selected_cells = sorted(set(expanded_cells))

zone_membership = candidate_counts[
    candidate_counts["h3"].isin(all_selected_cells)
].copy()
zone_membership["in_core"] = zone_membership["h3"].isin(core_cells)
zone_membership["in_expanded"] = zone_membership["h3"].isin(expanded_cells)
zone_membership.to_csv(
    OUTPUT_DIR / "soldier_field_h3_zone_membership_.csv", index=False
)

print("Exact Soldier Field H3 / 场馆精确 H3:", exact_stadium_h3)
print("Core cells / 核心区格子:", len(core_cells), core_cells)
print("Expanded cells / 扩展区格子:", len(expanded_cells), expanded_cells)
display(
    zone_membership[
        [
            "h3",
            "distance_to_stadium_km",
            "grid_distance_from_exact",
            "pickup_count",
            "dropoff_count",
            "total_trips",
            "in_core",
            "in_expanded",
        ]
    ]
)

# Plot true H3 outlines rather than centers only. / 绘制真实 H3 边界，而不是只画中心点。
fig, ax = plt.subplots(figsize=(10, 9))
for row in candidate_counts.itertuples(index=False):
    boundary = h3.cell_to_boundary(row.h3)
    polygon_lon = [point[1] for point in boundary] + [boundary[0][1]]
    polygon_lat = [point[0] for point in boundary] + [boundary[0][0]]

    if row.h3 in core_cells:
        color, linewidth, alpha = "#e67e22", 2.8, 0.9
    elif row.h3 in expanded_cells:
        color, linewidth, alpha = "#1976d2", 2.2, 0.8
    else:
        color, linewidth, alpha = "#9e9e9e", 0.9, 0.35

    ax.plot(
        polygon_lon,
        polygon_lat,
        color=color,
        linewidth=linewidth,
        alpha=alpha,
    )
    if row.h3 in expanded_cells:
        ax.text(
            row.lon,
            row.lat,
            f"{row.distance_to_stadium_km:.2f} km",
            fontsize=8,
            ha="center",
        )

ax.scatter(
    [SOLDIER_FIELD_LON],
    [SOLDIER_FIELD_LAT],
    marker="*",
    s=350,
    color="#c62828",
    edgecolor="black",
    label="Soldier Field",
    zorder=5,
)
ax.plot([], [], color="#e67e22", linewidth=3, label="Core zone / 核心区")
ax.plot([], [], color="#1976d2", linewidth=3, label="Expanded only / 扩展区")
ax.set_title("Distance-based Soldier Field H3 zones / 场馆 H3 距离分层")
ax.set_xlabel("Longitude / 经度")
ax.set_ylabel("Latitude / 纬度")
ax.legend()
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "01_distance_based_h3_zones.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## A2. 查询逐 H3 小时数据
## 2. Query hourly data for each H3

MatrixOne 按 `hour_start × H3` 聚合数据，再分别生成核心区、扩展区和单格结果，因此可以看到信号来自哪个格子。

The MatrixOne query keeps `hour_start × H3`, then builds core-zone, expanded-zone, and per-cell results so the source cell remains visible.

In [ ]:
# Cell 6 - Query per-H3 hourly pickup/dropoff / 查询逐 H3 小时客流

selected_sql = ", ".join(sql_string(cell) for cell in all_selected_cells)
pickup_cache = CACHE_DIR / "per_h3_hourly_pickups.csv.gz"
dropoff_cache = CACHE_DIR / "per_h3_hourly_dropoffs.csv.gz"

if (
    pickup_cache.exists()
    and dropoff_cache.exists()
    and not FORCE_REFRESH_HOURLY
):
    pickup_h3_hourly = pd.read_csv(
        pickup_cache, parse_dates=["hour_start"]
    )
    dropoff_h3_hourly = pd.read_csv(
        dropoff_cache, parse_dates=["hour_start"]
    )
    print("Loaded per-H3 hourly cache / 已读取逐 H3 小时缓存")
else:
    print("Query 1/2: per-H3 pickups / 查询 1/2：逐 H3 pickup")
    pickup_h3_hourly = query_df(f'''
    SELECT
        DATE_FORMAT(
            trip_start_timestamp, '%Y-%m-%d %H:00:00'
        ) AS hour_start,
        pickup_h3 AS h3,
        COUNT(*) AS pickup_count,
        SUM(COALESCE(trip_seconds, 0)) AS pickup_trip_seconds_sum
    FROM `{ANALYSIS_TABLE}`
    WHERE pickup_h3 IN ({selected_sql})
      AND trip_start_timestamp >= {sql_string(QUERY_START)}
      AND trip_start_timestamp < {sql_string(QUERY_END)}
      AND COALESCE(shared_trip_authorized, 0) = 0
    GROUP BY hour_start, pickup_h3
    ORDER BY hour_start, pickup_h3;
    ''')

    print("Query 2/2: per-H3 dropoffs / 查询 2/2：逐 H3 dropoff")
    dropoff_h3_hourly = query_df(f'''
    SELECT
        DATE_FORMAT(
            trip_end_timestamp, '%Y-%m-%d %H:00:00'
        ) AS hour_start,
        dropoff_h3 AS h3,
        COUNT(*) AS dropoff_count,
        SUM(COALESCE(trip_seconds, 0)) AS dropoff_trip_seconds_sum
    FROM `{ANALYSIS_TABLE}`
    WHERE dropoff_h3 IN ({selected_sql})
      AND trip_end_timestamp >= {sql_string(QUERY_START)}
      AND trip_end_timestamp < {sql_string(QUERY_END)}
      AND COALESCE(shared_trip_authorized, 0) = 0
    GROUP BY hour_start, dropoff_h3
    ORDER BY hour_start, dropoff_h3;
    ''')
    pickup_h3_hourly["hour_start"] = pd.to_datetime(
        pickup_h3_hourly["hour_start"]
    )
    dropoff_h3_hourly["hour_start"] = pd.to_datetime(
        dropoff_h3_hourly["hour_start"]
    )
    pickup_h3_hourly.to_csv(
        pickup_cache, index=False, compression="gzip"
    )
    dropoff_h3_hourly.to_csv(
        dropoff_cache, index=False, compression="gzip"
    )

for frame, columns in [
    (
        pickup_h3_hourly,
        ["pickup_count", "pickup_trip_seconds_sum"],
    ),
    (
        dropoff_h3_hourly,
        ["dropoff_count", "dropoff_trip_seconds_sum"],
    ),
]:
    frame["hour_start"] = pd.to_datetime(frame["hour_start"])
    frame["h3"] = frame["h3"].astype(str)
    for col in columns:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0)

full_hours = pd.date_range(
    QUERY_START,
    pd.Timestamp(QUERY_END) - pd.Timedelta(hours=1),
    freq="h",
)
full_index = pd.MultiIndex.from_product(
    [all_selected_cells, full_hours],
    names=["h3", "hour_start"],
)
cell_hourly = pd.DataFrame(index=full_index).reset_index()
cell_hourly = (
    cell_hourly
    .merge(pickup_h3_hourly, on=["h3", "hour_start"], how="left")
    .merge(dropoff_h3_hourly, on=["h3", "hour_start"], how="left")
)

numeric_cols = [
    "pickup_count",
    "pickup_trip_seconds_sum",
    "dropoff_count",
    "dropoff_trip_seconds_sum",
]
cell_hourly[numeric_cols] = cell_hourly[numeric_cols].fillna(0)
cell_hourly["weekday"] = cell_hourly["hour_start"].dt.weekday
cell_hourly["clock_hour"] = cell_hourly["hour_start"].dt.hour
cell_hourly["date"] = cell_hourly["hour_start"].dt.normalize()

# Aggregate nested zones from the same per-cell data. / 使用相同逐格数据构造核心区和扩展区。
zone_hourly_frames = []
for zone_name, zone_cells in ZONE_DEFINITIONS.items():
    zone_frame = (
        cell_hourly[cell_hourly["h3"].isin(zone_cells)]
        .groupby("hour_start", as_index=False)[numeric_cols]
        .sum()
    )
    zone_frame["zone_name"] = zone_name
    zone_frame["weekday"] = zone_frame["hour_start"].dt.weekday
    zone_frame["clock_hour"] = zone_frame["hour_start"].dt.hour
    zone_frame["date"] = zone_frame["hour_start"].dt.normalize()
    zone_hourly_frames.append(zone_frame)

zone_hourly = pd.concat(zone_hourly_frames, ignore_index=True)

print("Selected H3 cells / 选中的 H3 数量:", len(all_selected_cells))
print("Per-H3 hourly rows / 逐 H3 小时行数:", len(cell_hourly))
print("Zone-hourly rows / 区域小时行数:", len(zone_hourly))
display(cell_hourly.head())

## A3. 稳健基线、经验百分位与扩展事件窗口
## 3. Robust baseline, empirical percentile, and extended event window

每个目标小时与附近 70 天内相同星期几、相同钟点的非比赛小时比较。同时计算：

Each target hour is still matched with non-game hours having the same weekday and clock hour within 70 days. The revision adds:

- `robust_z = (actual - median) / (1.4826 × MAD)`  
  使用中位数和 MAD，降低极端控制日的影响。
- `empirical_percentile`  
  实际值在控制组中的经验排名。例如 100 表示高于全部控制小时。
- `p05 / p95`  
  控制组的 5% 和 95% 分位数，不依赖正态分布假设。

主图使用稳健 z-score；普通 z-score仍然保留，便于与第一版对照。

In [ ]:
# Cell 7 - Calculate robust sensitivity / 计算稳健敏感度

excluded_event_hours = set()
for kickoff_hour in games["kickoff_hour"]:
    for rel_hour in EXCLUSION_RELATIVE_HOURS:
        excluded_event_hours.add(
            kickoff_hour + pd.Timedelta(hours=rel_hour)
        )

# Store all analysis units in one dictionary. / 把区域和单格数据统一放进字典。
unit_frames = {}
for zone_name in ZONE_DEFINITIONS:
    key = f"zone:{zone_name}"
    unit_frames[key] = (
        zone_hourly[zone_hourly["zone_name"] == zone_name]
        .sort_values("hour_start")
        .reset_index(drop=True)
    )
for cell in all_selected_cells:
    key = f"cell:{cell}"
    unit_frames[key] = (
        cell_hourly[cell_hourly["h3"] == cell]
        .sort_values("hour_start")
        .reset_index(drop=True)
    )

unit_lookup = {
    key: frame.set_index("hour_start")
    for key, frame in unit_frames.items()
}
control_cache = {}


def calculate_control_stats(unit_key, target_hour, value_column):
    '''
    Calculate matched baseline statistics for one unit and target hour.
    为一个区域和目标小时计算匹配基线。
    '''
    target_hour = pd.Timestamp(target_hour)
    cache_key = (unit_key, target_hour, value_column)
    if cache_key in control_cache:
        return control_cache[cache_key]

    frame = unit_frames[unit_key]
    day_distance = (
        frame["date"] - target_hour.normalize()
    ).abs().dt.days
    mask = (
        (frame["weekday"] == target_hour.weekday())
        & (frame["clock_hour"] == target_hour.hour)
        & (day_distance <= CONTROL_WINDOW_DAYS)
        & (~frame["hour_start"].isin(excluded_event_hours))
    )
    controls = frame.loc[mask, value_column].dropna().astype(float)

    if len(controls) < MIN_CONTROL_HOURS:
        fallback = (
            (frame["weekday"] == target_hour.weekday())
            & (frame["clock_hour"] == target_hour.hour)
            & (~frame["hour_start"].isin(excluded_event_hours))
        )
        controls = frame.loc[fallback, value_column].dropna().astype(float)

    mean_value = controls.mean()
    std_value = controls.std(ddof=1)
    median_value = controls.median()
    mad_value = (controls - median_value).abs().median()
    robust_std = 1.4826 * mad_value

    result = {
        "control_n": int(len(controls)),
        "expected_mean": mean_value,
        "expected_std": std_value,
        "expected_median": median_value,
        "expected_mad": mad_value,
        "expected_robust_std": robust_std,
        "expected_p05": controls.quantile(0.05),
        "expected_p95": controls.quantile(0.95),
        "controls": controls.to_numpy(),
    }
    control_cache[cache_key] = result
    return result


def calculate_target_metrics(unit_key, target_hour, flow_name):
    '''
    Calculate actual, lift, z-scores, and percentile for one target.
    计算一个目标小时的实际值、涨幅、z-score 和经验百分位。
    '''
    target_hour = pd.Timestamp(target_hour)
    value_column = f"{flow_name}_count"
    lookup = unit_lookup[unit_key]
    if target_hour not in lookup.index:
        return None

    actual = float(lookup.loc[target_hour, value_column])
    stats = calculate_control_stats(
        unit_key, target_hour, value_column
    )
    mean_value = stats["expected_mean"]
    median_value = stats["expected_median"]
    std_value = stats["expected_std"]
    robust_std = stats["expected_robust_std"]
    controls = stats["controls"]

    lift = (
        actual / mean_value - 1
        if pd.notna(mean_value) and mean_value > 0
        else np.nan
    )
    z_score = (
        (actual - mean_value) / std_value
        if pd.notna(std_value) and std_value > 0
        else np.nan
    )
    robust_z = (
        (actual - median_value) / robust_std
        if pd.notna(robust_std) and robust_std > 0
        else np.nan
    )
    percentile = (
        100 * np.mean(controls <= actual)
        if len(controls)
        else np.nan
    )

    return {
        f"{flow_name}_actual": actual,
        f"{flow_name}_expected_mean": mean_value,
        f"{flow_name}_expected_median": median_value,
        f"{flow_name}_expected_p05": stats["expected_p05"],
        f"{flow_name}_expected_p95": stats["expected_p95"],
        f"{flow_name}_control_n": stats["control_n"],
        f"{flow_name}_lift_pct": lift * 100,
        f"{flow_name}_z": z_score,
        f"{flow_name}_abs_z": abs(z_score)
        if pd.notna(z_score) else np.nan,
        f"{flow_name}_robust_z": robust_z,
        f"{flow_name}_abs_robust_z": abs(robust_z)
        if pd.notna(robust_z) else np.nan,
        f"{flow_name}_empirical_percentile": percentile,
    }


def build_event_sensitivity(unit_key, unit_type, unit_id):
    '''
    Build all game-relative rows for one zone or H3 cell.
    为一个区域或 H3 构造全部比赛相对小时记录。
    '''
    rows = []
    for game in games.itertuples(index=False):
        for relative_hour in RELATIVE_HOURS:
            target_hour = (
                game.kickoff_hour
                + pd.Timedelta(hours=relative_hour)
            )
            row = {
                "unit_key": unit_key,
                "unit_type": unit_type,
                "unit_id": unit_id,
                "game_id": game.game_id,
                "game_label": game.game_label,
                "season": game.season,
                "week": game.week,
                "opponent": game.opponent,
                "scheduled_kickoff_local": game.scheduled_kickoff_local,
                "estimated_game_end_local": game.estimated_game_end_local,
                "is_holiday_period": game.is_holiday_period,
                "event_hour_start": target_hour,
                "relative_hour": relative_hour,
            }
            valid = True
            for flow_name in ["pickup", "dropoff"]:
                metrics = calculate_target_metrics(
                    unit_key, target_hour, flow_name
                )
                if metrics is None:
                    valid = False
                    break
                row.update(metrics)
            if valid:
                rows.append(row)
    return pd.DataFrame(rows)


zone_sensitivity_frames = []
for zone_name in ZONE_DEFINITIONS:
    zone_sensitivity_frames.append(
        build_event_sensitivity(
            f"zone:{zone_name}", "zone", zone_name
        )
    )
zone_sensitivity = pd.concat(
    zone_sensitivity_frames, ignore_index=True
)

cell_sensitivity_frames = []
for cell in all_selected_cells:
    cell_sensitivity_frames.append(
        build_event_sensitivity(f"cell:{cell}", "cell", cell)
    )
cell_sensitivity = pd.concat(
    cell_sensitivity_frames, ignore_index=True
)

primary_sensitivity = zone_sensitivity[
    zone_sensitivity["unit_id"] == PRIMARY_ZONE
].copy()

zone_sensitivity.to_csv(
    OUTPUT_DIR / "bears_zone_event_sensitivity_.csv", index=False
)
cell_sensitivity.to_csv(
    OUTPUT_DIR / "bears_per_h3_event_sensitivity_.csv", index=False
)

print("Zone event rows / 区域事件记录数:", len(zone_sensitivity))
print("Per-H3 event rows / 逐 H3 事件记录数:", len(cell_sensitivity))
print(
    "Primary-zone rows / 主区域记录数:",
    len(primary_sensitivity),
    "expected / 应有:",
    len(games) * len(RELATIVE_HOURS),
)
display(
    primary_sensitivity[
        [
            "game_label",
            "event_hour_start",
            "relative_hour",
            "pickup_actual",
            "pickup_expected_median",
            "pickup_robust_z",
            "pickup_empirical_percentile",
            "dropoff_actual",
            "dropoff_expected_median",
            "dropoff_robust_z",
            "dropoff_empirical_percentile",
        ]
    ].head(15)
)

In [ ]:
# Cell 8 - Detect window boundaries / 检查窗口是否仍被截断

timing_summary = (
    primary_sensitivity
    .groupby("relative_hour", as_index=False)
    .agg(
        pickup_actual=("pickup_actual", "mean"),
        pickup_expected=("pickup_expected_median", "mean"),
        pickup_mean_robust_z=("pickup_robust_z", "mean"),
        pickup_median_robust_z=("pickup_robust_z", "median"),
        pickup_share_z_ge_3=(
            "pickup_robust_z",
            lambda s: np.mean(s >= ROBUST_Z_THRESHOLD),
        ),
        dropoff_actual=("dropoff_actual", "mean"),
        dropoff_expected=("dropoff_expected_median", "mean"),
        dropoff_mean_robust_z=("dropoff_robust_z", "mean"),
        dropoff_median_robust_z=("dropoff_robust_z", "median"),
        dropoff_share_z_ge_3=(
            "dropoff_robust_z",
            lambda s: np.mean(s >= ROBUST_Z_THRESHOLD),
        ),
    )
    .sort_values("relative_hour")
)
timing_summary.to_csv(
    OUTPUT_DIR / "bears_extended_timing_summary_.csv", index=False
)

earliest = timing_summary.iloc[0]
latest = timing_summary.iloc[-1]
LEFT_WINDOW_TRUNCATED = (
    earliest["dropoff_mean_robust_z"] >= ROBUST_Z_THRESHOLD
)
RIGHT_WINDOW_TRUNCATED = (
    latest["pickup_mean_robust_z"] >= ROBUST_Z_THRESHOLD
)

print(
    "Left boundary still elevated / 左边界仍处于高位:",
    LEFT_WINDOW_TRUNCATED,
)
print(
    "Right boundary still elevated / 右边界仍处于高位:",
    RIGHT_WINDOW_TRUNCATED,
)
print(
    "At -6h mean dropoff robust z / -6 小时平均 dropoff robust z:",
    round(earliest["dropoff_mean_robust_z"], 3),
)
print(
    "At +8h mean pickup robust z / +8 小时平均 pickup robust z:",
    round(latest["pickup_mean_robust_z"], 3),
)
display(timing_summary)

## A4. 扩展时间曲线
## 4. Extended timing curves

黑色竖线是计划开球所在小时。绿色虚线是统一估算的比赛结束位置 `+3.25`，只作为参考。

The black line marks scheduled kickoff. The green dashed line at `+3.25` is only an estimated end-time reference.

如果最左边的 dropoff 已经明显高于预期，窗口仍然太短；如果最右边的 pickup 仍然明显高于预期，赛后窗口仍然太短。

In [ ]:
# Cell 9 - Plot extended observed vs expected curves / 绘制扩展后的实际值与预期值曲线

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)

axes[0].plot(
    timing_summary["relative_hour"],
    timing_summary["pickup_actual"],
    marker="o",
    linewidth=2.4,
    color="#1565c0",
    label="Observed pickup / 实际 pickup",
)
axes[0].plot(
    timing_summary["relative_hour"],
    timing_summary["pickup_expected"],
    marker="o",
    linestyle="--",
    color="#90caf9",
    label="Expected median / 预期中位数",
)
axes[0].set_title("Extended pickup curve / 扩展 pickup 曲线")
axes[0].set_ylabel("Trips per hour / 每小时订单")

axes[1].plot(
    timing_summary["relative_hour"],
    timing_summary["dropoff_actual"],
    marker="o",
    linewidth=2.4,
    color="#c62828",
    label="Observed dropoff / 实际 dropoff",
)
axes[1].plot(
    timing_summary["relative_hour"],
    timing_summary["dropoff_expected"],
    marker="o",
    linestyle="--",
    color="#ef9a9a",
    label="Expected median / 预期中位数",
)
axes[1].set_title("Extended dropoff curve / 扩展 dropoff 曲线")

for ax in axes:
    ax.axvline(
        0,
        color="black",
        linestyle=":",
        linewidth=2,
        label="Kickoff hour / 开球小时",
    )
    ax.axvline(
        ESTIMATED_GAME_DURATION_HOURS,
        color="#2e7d32",
        linestyle="--",
        linewidth=1.8,
        label="Estimated end / 估算结束",
    )
    ax.axvspan(min(RELATIVE_HOURS), 0, color="#fff3cd", alpha=0.35)
    ax.axvspan(0, max(RELATIVE_HOURS), color="#e3f2fd", alpha=0.22)
    ax.set_xlabel("Calendar hours from kickoff / 距开球自然小时")
    ax.set_xticks(RELATIVE_HOURS)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "02_extended_observed_vs_expected.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Cell 10 - Plot robust sensitivity and consistency / 绘制稳健敏感度与跨比赛一致性

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)

axes[0].plot(
    timing_summary["relative_hour"],
    timing_summary["pickup_mean_robust_z"],
    marker="o",
    color="#1565c0",
    label="Pickup mean robust z",
)
axes[0].plot(
    timing_summary["relative_hour"],
    timing_summary["dropoff_mean_robust_z"],
    marker="o",
    color="#c62828",
    label="Dropoff mean robust z",
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].axhline(
    ROBUST_Z_THRESHOLD,
    color="#7b1fa2",
    linestyle="--",
    label="z = 3",
)
axes[0].set_title("Mean robust sensitivity / 平均稳健敏感度")
axes[0].set_ylabel("Robust z-score")

axes[1].plot(
    timing_summary["relative_hour"],
    timing_summary["pickup_share_z_ge_3"] * 100,
    marker="o",
    color="#1565c0",
    label="Pickup games with z≥3",
)
axes[1].plot(
    timing_summary["relative_hour"],
    timing_summary["dropoff_share_z_ge_3"] * 100,
    marker="o",
    color="#c62828",
    label="Dropoff games with z≥3",
)
axes[1].set_ylim(-3, 103)
axes[1].set_title("Share of games with robust z≥3 / 达到阈值的比赛占比")
axes[1].set_ylabel("Games / 比赛占比 (%)")

for ax in axes:
    ax.axvline(0, color="black", linestyle=":", linewidth=2)
    ax.axvline(
        ESTIMATED_GAME_DURATION_HOURS,
        color="#2e7d32",
        linestyle="--",
        linewidth=1.8,
    )
    ax.set_xlabel("Calendar hours from kickoff / 距开球自然小时")
    ax.set_xticks(RELATIVE_HOURS)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "03_robust_sensitivity_and_consistency.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## A5. 逐 H3 与空间距离分析
## 5. Per-H3 and distance analysis

每行对应一个有效 H3。左图看 pickup robust z，右图看 dropoff robust z。标题同时显示该格中心距 Soldier Field 的距离。

Each row represents one observed H3. The left chart shows pickup robust z and the right chart shows dropoff robust z. Titles include distance to Soldier Field.

如果最近格子的信号最强并随距离衰减，空间解释更可信；如果只有最远格子很强，需要检查其他场馆或活动干扰。

In [ ]:
# Cell 11 - Plot per-H3 event curves and distance sensitivity / 绘制逐 H3 曲线与距离敏感度

cell_meta = zone_membership.set_index("h3")
per_cell_curve = (
    cell_sensitivity
    .groupby(["unit_id", "relative_hour"], as_index=False)
    .agg(
        pickup_mean_robust_z=("pickup_robust_z", "mean"),
        dropoff_mean_robust_z=("dropoff_robust_z", "mean"),
    )
)

n_cells = len(all_selected_cells)
fig, axes = plt.subplots(
    n_cells,
    2,
    figsize=(17, max(4, 3.8 * n_cells)),
    sharex=True,
)
if n_cells == 1:
    axes = np.array([axes])

for row_idx, cell in enumerate(all_selected_cells):
    data = per_cell_curve[per_cell_curve["unit_id"] == cell]
    distance = cell_meta.loc[cell, "distance_to_stadium_km"]

    axes[row_idx, 0].plot(
        data["relative_hour"],
        data["pickup_mean_robust_z"],
        marker="o",
        color="#1565c0",
    )
    axes[row_idx, 0].set_title(
        f"{cell} pickup — {distance:.2f} km"
    )
    axes[row_idx, 0].set_ylabel("Mean robust z")

    axes[row_idx, 1].plot(
        data["relative_hour"],
        data["dropoff_mean_robust_z"],
        marker="o",
        color="#c62828",
    )
    axes[row_idx, 1].set_title(
        f"{cell} dropoff — {distance:.2f} km"
    )

    for ax in axes[row_idx]:
        ax.axhline(0, color="black", linewidth=1)
        ax.axhline(3, color="#7b1fa2", linestyle="--", alpha=0.7)
        ax.axvline(0, color="black", linestyle=":", linewidth=1.8)
        ax.axvline(
            ESTIMATED_GAME_DURATION_HOURS,
            color="#2e7d32",
            linestyle="--",
            linewidth=1.5,
        )
        ax.set_xticks(RELATIVE_HOURS)
        ax.set_xlabel("Hours from kickoff / 距开球小时")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "04_per_h3_robust_sensitivity.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

spatial_summary_rows = []
for cell in all_selected_cells:
    data = cell_sensitivity[cell_sensitivity["unit_id"] == cell]
    arrival = data[data["relative_hour"].isin(ARRIVAL_SCORE_HOURS)]
    exit_data = data[data["relative_hour"].isin(EXIT_SCORE_HOURS)]
    spatial_summary_rows.append(
        {
            "h3": cell,
            "distance_to_stadium_km": cell_meta.loc[
                cell, "distance_to_stadium_km"
            ],
            "in_core": bool(cell_meta.loc[cell, "in_core"]),
            "arrival_dropoff_mean_robust_z": arrival[
                "dropoff_robust_z"
            ].mean(),
            "arrival_dropoff_peak_robust_z": arrival[
                "dropoff_robust_z"
            ].max(),
            "exit_pickup_mean_robust_z": exit_data[
                "pickup_robust_z"
            ].mean(),
            "exit_pickup_peak_robust_z": exit_data[
                "pickup_robust_z"
            ].max(),
        }
    )
spatial_summary = pd.DataFrame(spatial_summary_rows).sort_values(
    "distance_to_stadium_km"
)
spatial_summary.to_csv(
    OUTPUT_DIR / "per_h3_spatial_sensitivity_summary_.csv",
    index=False,
)
display(spatial_summary)

In [ ]:
# Cell 12 - Compare core and expanded zones / 对比核心区与扩展区

zone_curve = (
    zone_sensitivity
    .groupby(["unit_id", "relative_hour"], as_index=False)
    .agg(
        pickup_mean_robust_z=("pickup_robust_z", "mean"),
        dropoff_mean_robust_z=("dropoff_robust_z", "mean"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)
colors = {"core": "#e67e22", "expanded": "#1976d2"}

for zone_name in ZONE_DEFINITIONS:
    data = zone_curve[zone_curve["unit_id"] == zone_name]
    axes[0].plot(
        data["relative_hour"],
        data["pickup_mean_robust_z"],
        marker="o",
        color=colors[zone_name],
        label=zone_name,
    )
    axes[1].plot(
        data["relative_hour"],
        data["dropoff_mean_robust_z"],
        marker="o",
        color=colors[zone_name],
        label=zone_name,
    )

axes[0].set_title("Pickup: core vs expanded / pickup 区域对比")
axes[1].set_title("Dropoff: core vs expanded / dropoff 区域对比")
for ax in axes:
    ax.axhline(0, color="black", linewidth=1)
    ax.axhline(3, color="#7b1fa2", linestyle="--", alpha=0.7)
    ax.axvline(0, color="black", linestyle=":", linewidth=2)
    ax.axvline(
        ESTIMATED_GAME_DURATION_HOURS,
        color="#2e7d32",
        linestyle="--",
        linewidth=1.5,
    )
    ax.set_xticks(RELATIVE_HOURS)
    ax.set_xlabel("Hours from kickoff / 距开球小时")
    ax.set_ylabel("Mean robust z")
    ax.legend()

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "05_core_vs_expanded_zone.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## A6. 逐场热力图与比赛排名
## 6. Per-game heatmaps and game ranking

热力图使用核心区和稳健 z-score。每一行是一场比赛，每列是 `-6...+8` 的相对小时。

The heatmaps use the core zone and robust z-score. Each row is one game and each column is a relative hour from `-6...+8`.

排名把“到场 dropoff 信号”和“离场附近 pickup 信号”分开计算，不再使用模糊的 in-game/postgame 标签。

In [ ]:
# Cell 13 - Extended per-game heatmaps / 扩展逐场热力图

game_order = games.sort_values("scheduled_kickoff_local")[
    "game_label"
].tolist()
pickup_heat = (
    primary_sensitivity
    .pivot(
        index="game_label",
        columns="relative_hour",
        values="pickup_robust_z",
    )
    .reindex(game_order)
)
dropoff_heat = (
    primary_sensitivity
    .pivot(
        index="game_label",
        columns="relative_hour",
        values="dropoff_robust_z",
    )
    .reindex(game_order)
)

finite_values = np.concatenate(
    [pickup_heat.to_numpy().ravel(), dropoff_heat.to_numpy().ravel()]
)
finite_values = finite_values[np.isfinite(finite_values)]
color_limit = (
    max(3, np.nanpercentile(np.abs(finite_values), 95))
    if len(finite_values) else 3
)

fig, axes = plt.subplots(1, 2, figsize=(20, 11), sharey=True)
sns.heatmap(
    pickup_heat,
    ax=axes[0],
    cmap="RdBu_r",
    center=0,
    vmin=-color_limit,
    vmax=color_limit,
    annot=True,
    fmt=".1f",
    cbar_kws={"label": "pickup robust z"},
)
axes[0].set_title("Core-zone pickup robust z / 核心区 pickup")
axes[0].set_xlabel("Hours from kickoff / 距开球小时")
axes[0].set_ylabel("Game / 比赛")

sns.heatmap(
    dropoff_heat,
    ax=axes[1],
    cmap="RdBu_r",
    center=0,
    vmin=-color_limit,
    vmax=color_limit,
    annot=True,
    fmt=".1f",
    cbar_kws={"label": "dropoff robust z"},
)
axes[1].set_title("Core-zone dropoff robust z / 核心区 dropoff")
axes[1].set_xlabel("Hours from kickoff / 距开球小时")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "06_extended_per_game_heatmaps.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Cell 14 - Build arrival and exit sensitivity ranking / 构造到场与离场敏感度排名

game_summary_rows = []
for game_id, group in primary_sensitivity.groupby("game_id", sort=False):
    first = group.iloc[0]
    arrival = group[group["relative_hour"].isin(ARRIVAL_SCORE_HOURS)]
    exit_data = group[group["relative_hour"].isin(EXIT_SCORE_HOURS)]

    arrival_peak_row = arrival.loc[
        arrival["dropoff_robust_z"].idxmax()
    ]
    exit_peak_row = exit_data.loc[
        exit_data["pickup_robust_z"].idxmax()
    ]

    game_summary_rows.append(
        {
            "game_id": game_id,
            "game_label": first["game_label"],
            "scheduled_kickoff_local": first[
                "scheduled_kickoff_local"
            ],
            "is_holiday_period": first["is_holiday_period"],
            "arrival_dropoff_mean_robust_z": arrival[
                "dropoff_robust_z"
            ].mean(),
            "arrival_dropoff_peak_robust_z": arrival_peak_row[
                "dropoff_robust_z"
            ],
            "arrival_peak_relative_hour": arrival_peak_row[
                "relative_hour"
            ],
            "exit_pickup_mean_robust_z": exit_data[
                "pickup_robust_z"
            ].mean(),
            "exit_pickup_peak_robust_z": exit_peak_row[
                "pickup_robust_z"
            ],
            "exit_peak_relative_hour": exit_peak_row[
                "relative_hour"
            ],
        }
    )

game_summary = pd.DataFrame(game_summary_rows)
game_summary["combined_event_score"] = (
    game_summary["arrival_dropoff_mean_robust_z"]
    + game_summary["exit_pickup_mean_robust_z"]
) / 2
game_summary = game_summary.sort_values(
    "combined_event_score", ascending=False
).reset_index(drop=True)
game_summary.to_csv(
    OUTPUT_DIR / "bears_game_sensitivity_summary_.csv", index=False
)
display(game_summary)

In [ ]:
# Cell 15 - Detailed curves for the six strongest games / 绘制六场最强比赛的详细曲线

top_game_ids = game_summary.head(min(6, len(game_summary)))[
    "game_id"
].tolist()
n_games = len(top_game_ids)
fig, axes = plt.subplots(
    n_games,
    2,
    figsize=(18, 3.8 * n_games),
    sharex=True,
)
if n_games == 1:
    axes = np.array([axes])

for row_idx, game_id in enumerate(top_game_ids):
    data = primary_sensitivity[
        primary_sensitivity["game_id"] == game_id
    ].sort_values("relative_hour")
    label = data["game_label"].iloc[0]

    axes[row_idx, 0].plot(
        data["relative_hour"],
        data["pickup_actual"],
        marker="o",
        color="#1565c0",
        label="Actual / 实际",
    )
    axes[row_idx, 0].plot(
        data["relative_hour"],
        data["pickup_expected_median"],
        marker="o",
        linestyle="--",
        color="#90caf9",
        label="Expected median / 预期中位数",
    )
    axes[row_idx, 0].set_title(f"{label} — pickup")
    axes[row_idx, 0].set_ylabel("Trips per hour / 每小时订单")

    axes[row_idx, 1].plot(
        data["relative_hour"],
        data["dropoff_actual"],
        marker="o",
        color="#c62828",
        label="Actual / 实际",
    )
    axes[row_idx, 1].plot(
        data["relative_hour"],
        data["dropoff_expected_median"],
        marker="o",
        linestyle="--",
        color="#ef9a9a",
        label="Expected median / 预期中位数",
    )
    axes[row_idx, 1].set_title(f"{label} — dropoff")

    for ax in axes[row_idx]:
        ax.axvline(0, color="black", linestyle=":", linewidth=1.8)
        ax.axvline(
            ESTIMATED_GAME_DURATION_HOURS,
            color="#2e7d32",
            linestyle="--",
            linewidth=1.5,
        )
        ax.set_xticks(RELATIVE_HOURS)
        ax.set_xlabel("Hours from kickoff / 距开球小时")
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "07_top_games_extended_detail.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## A7. Matched placebo days
## 7. 匹配的非比赛日对照

Placebo 测试把相同星期几、相同开球钟点附近的非比赛日临时当成“假比赛日”，然后用完全相同的方法计算到场和离场分数。

The placebo test treats matched non-game dates with the same weekday and kickoff clock hour as fake game days and calculates the same arrival and exit scores.

如果真实比赛分数明显高于 placebo，说明结果不只是星期规律或普通日常波动。

In [ ]:
# Cell 16 - Compare real games with matched placebo days / 将真实比赛与匹配的非比赛日比较

primary_unit_key = f"zone:{PRIMARY_ZONE}"
primary_frame = unit_frames[primary_unit_key]
game_dates = set(games["kickoff_hour"].dt.normalize())


def build_signature_for_anchor(anchor_hour):
    '''
    Build arrival and exit scores around a real or placebo anchor.
    围绕真实或 placebo 时间点构造到场和离场分数。
    '''
    rows = []
    for relative_hour in RELATIVE_HOURS:
        target_hour = anchor_hour + pd.Timedelta(hours=relative_hour)
        pickup = calculate_target_metrics(
            primary_unit_key, target_hour, "pickup"
        )
        dropoff = calculate_target_metrics(
            primary_unit_key, target_hour, "dropoff"
        )
        if pickup is None or dropoff is None:
            continue
        rows.append(
            {
                "relative_hour": relative_hour,
                **pickup,
                **dropoff,
            }
        )
    frame = pd.DataFrame(rows)
    if len(frame) != len(RELATIVE_HOURS):
        return None

    arrival = frame[
        frame["relative_hour"].isin(ARRIVAL_SCORE_HOURS)
    ]
    exit_data = frame[
        frame["relative_hour"].isin(EXIT_SCORE_HOURS)
    ]
    return {
        "arrival_score": arrival["dropoff_robust_z"].mean(),
        "arrival_peak": arrival["dropoff_robust_z"].max(),
        "exit_score": exit_data["pickup_robust_z"].mean(),
        "exit_peak": exit_data["pickup_robust_z"].max(),
    }


real_signature_rows = []
placebo_rows = []

for game in games.itertuples(index=False):
    real_signature = build_signature_for_anchor(game.kickoff_hour)
    real_signature_rows.append(
        {
            "game_id": game.game_id,
            "game_label": game.game_label,
            "anchor_hour": game.kickoff_hour,
            "sample_type": "real_game",
            **real_signature,
        }
    )

    # Same weekday and clock hour, every seven days within ±70 days. / 在前后 70 天内选择相同星期几和钟点。
    for day_offset in range(
        -CONTROL_WINDOW_DAYS,
        CONTROL_WINDOW_DAYS + 1,
        7,
    ):
        if day_offset == 0:
            continue
        anchor = game.kickoff_hour + pd.Timedelta(days=day_offset)
        if anchor.normalize() in game_dates:
            continue
        if (
            anchor + pd.Timedelta(hours=min(RELATIVE_HOURS))
            < pd.Timestamp(QUERY_START)
            or anchor + pd.Timedelta(hours=max(RELATIVE_HOURS) + 1)
            >= pd.Timestamp(QUERY_END)
        ):
            continue
        if any(
            anchor + pd.Timedelta(hours=rel)
            in excluded_event_hours
            for rel in RELATIVE_HOURS
        ):
            continue

        signature = build_signature_for_anchor(anchor)
        if signature is None:
            continue
        placebo_rows.append(
            {
                "matched_game_id": game.game_id,
                "matched_game_label": game.game_label,
                "anchor_hour": anchor,
                "sample_type": "placebo",
                **signature,
            }
        )

real_signatures = pd.DataFrame(real_signature_rows)
placebo_signatures = pd.DataFrame(placebo_rows)

placebo_thresholds = (
    placebo_signatures
    .groupby("matched_game_id", as_index=False)
    .agg(
        placebo_arrival_p95=("arrival_score", lambda s: s.quantile(0.95)),
        placebo_exit_p95=("exit_score", lambda s: s.quantile(0.95)),
        placebo_n=("arrival_score", "count"),
    )
)
placebo_comparison = real_signatures.merge(
    placebo_thresholds,
    left_on="game_id",
    right_on="matched_game_id",
    how="left",
)
placebo_comparison["arrival_exceeds_placebo_p95"] = (
    placebo_comparison["arrival_score"]
    > placebo_comparison["placebo_arrival_p95"]
)
placebo_comparison["exit_exceeds_placebo_p95"] = (
    placebo_comparison["exit_score"]
    > placebo_comparison["placebo_exit_p95"]
)

real_signatures.to_csv(
    OUTPUT_DIR / "real_game_signatures_.csv", index=False
)
placebo_signatures.to_csv(
    OUTPUT_DIR / "matched_placebo_signatures_.csv", index=False
)
placebo_comparison.to_csv(
    OUTPUT_DIR / "real_vs_placebo_comparison_.csv", index=False
)

plot_real = real_signatures[
    ["arrival_score", "exit_score"]
].copy()
plot_real["sample_type"] = "Real Bears game"
plot_placebo = placebo_signatures[
    ["arrival_score", "exit_score"]
].copy()
plot_placebo["sample_type"] = "Matched non-game day"
plot_data = pd.concat([plot_real, plot_placebo], ignore_index=True)
plot_long = plot_data.melt(
    id_vars="sample_type",
    value_vars=["arrival_score", "exit_score"],
    var_name="metric",
    value_name="robust_z_score",
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, metric, title in [
    (
        axes[0],
        "arrival_score",
        "Arrival dropoff score / 到场 dropoff 分数",
    ),
    (
        axes[1],
        "exit_score",
        "Exit pickup score / 离场 pickup 分数",
    ),
]:
    subset = plot_long[plot_long["metric"] == metric]
    sns.boxplot(
        data=subset,
        x="sample_type",
        y="robust_z_score",
        ax=ax,
        color="#bbdefb",
    )
    sns.stripplot(
        data=subset[subset["sample_type"] == "Real Bears game"],
        x="sample_type",
        y="robust_z_score",
        ax=ax,
        color="#c62828",
        size=5,
        jitter=0.15,
    )
    ax.axhline(3, color="#7b1fa2", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Mean robust z")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "08_real_games_vs_placebo.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

print(
    "Arrival hit rate above matched placebo p95 / "
    "到场分数超过匹配 placebo 95% 阈值:",
    f"{placebo_comparison['arrival_exceeds_placebo_p95'].mean():.1%}",
)
print(
    "Exit hit rate above matched placebo p95 / "
    "离场分数超过匹配 placebo 95% 阈值:",
    f"{placebo_comparison['exit_exceeds_placebo_p95'].mean():.1%}",
)
display(
    placebo_comparison[
        [
            "game_label",
            "arrival_score",
            "placebo_arrival_p95",
            "arrival_exceeds_placebo_p95",
            "exit_score",
            "placebo_exit_p95",
            "exit_exceeds_placebo_p95",
            "placebo_n",
        ]
    ]
)

## A8. 运行检查与输出
## 8. Run checks and outputs

结构检查通过只表示数据行数、区域和控制样本完整。事件因果解释仍需结合实际比赛结束时间、天气、其他 Soldier Field 活动和附近场馆活动。

Passing structural checks means the rows, zones, and controls are complete. Causal interpretation still requires actual game-end times, weather, other Soldier Field events, and nearby venue events.

In [ ]:
# Cell 17 - Final validation / 最终验证

expected_primary_rows = len(games) * len(RELATIVE_HOURS)
checks = pd.DataFrame(
    [
        {
            "check": "regular_season_home_games",
            "value": len(games),
            "expected": 17,
            "passed": len(games) == 17,
        },
        {
            "check": "primary_zone_event_rows",
            "value": len(primary_sensitivity),
            "expected": expected_primary_rows,
            "passed": len(primary_sensitivity) == expected_primary_rows,
        },
        {
            "check": "core_zone_cells",
            "value": len(core_cells),
            "expected": ">= 1",
            "passed": len(core_cells) >= 1,
        },
        {
            "check": "expanded_contains_core",
            "value": set(core_cells).issubset(set(expanded_cells)),
            "expected": True,
            "passed": set(core_cells).issubset(set(expanded_cells)),
        },
        {
            "check": "minimum_pickup_controls",
            "value": int(primary_sensitivity["pickup_control_n"].min()),
            "expected": f">= {MIN_CONTROL_HOURS}",
            "passed": (
                primary_sensitivity["pickup_control_n"].min()
                >= MIN_CONTROL_HOURS
            ),
        },
        {
            "check": "minimum_dropoff_controls",
            "value": int(primary_sensitivity["dropoff_control_n"].min()),
            "expected": f">= {MIN_CONTROL_HOURS}",
            "passed": (
                primary_sensitivity["dropoff_control_n"].min()
                >= MIN_CONTROL_HOURS
            ),
        },
        {
            "check": "placebo_rows",
            "value": len(placebo_signatures),
            "expected": "> 0",
            "passed": len(placebo_signatures) > 0,
        },
    ]
)
checks.to_csv(OUTPUT_DIR / "run_validation_checks_.csv", index=False)
display(checks)

print("Output folder / 输出目录:", OUTPUT_DIR)
print("\nGenerated files / 已生成文件:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print("-", path.name)

if checks["passed"].all():
    print(
        "\nAll structural checks passed. / "
        "所有结构检查通过。"
    )
else:
    print(
        "\nSome structural checks require review. / "
        "部分结构检查需要确认。"
    )

print(
    "\nWindow boundary result / 窗口边界结果:",
    {
        "left_window_truncated": bool(LEFT_WINDOW_TRUNCATED),
        "right_window_truncated": bool(RIGHT_WINDOW_TRUNCATED),
    },
)

> **Continuation / 承接说明：** This section is appended after the completed 2022-2023 Bears home-game sensitivity analysis. It runs two independent 2024 tests: known-schedule hourly-demand forecasting and schedule-hidden blind event detection.
> **承接说明：** 本节直接接在已完成的 2022-2023 Bears 主场比赛敏感度分析之后，包含两个独立的 2024 测试：已知赛程的逐小时客流预测，以及隐藏赛程的盲测事件识别。

# Part B. 2024 known-schedule forecast and blind detection  
# 第二部分：2024 已知赛程预测与盲测
# 07. Chicago Bears 2024 Known-Schedule Forecast and Blind Test

## 测试目标 / Test goals

本 Notebook 使用 2022–2023 年 Soldier Field 周边客流学习“正常小时基线”和 Bears 主场比赛事件模板，然后在完全未参与训练的 2024 年做两个测试。

This notebook learns a normal hourly baseline and a Bears home-game event template from 2022–2023, then tests both on the unseen year 2024.

#### B1. 给出比赛时间 / Test 1: known schedule

已知 2024 年 Soldier Field 常规赛日期和计划开球时间。模型根据 2022–2023 模板预测比赛前后每小时 pickup/dropoff，再与实际 2024 客流比较。

The 2024 Soldier Field game date and scheduled kickoff are provided. The model predicts hourly pickup/dropoff around each game and compares them with actual 2024 flow.

#### B2. 不给出比赛时间 / Test 2: blind detection

候选生成器扫描 2024 全年客流，不读取 2024 赛程。它寻找“赛前 dropoff 上升、数小时后 pickup 上升”的时序形状，并把最可能的锚点小时当作预测开球时间。候选结果保存并生成 SHA-256 后，才加载真实比赛信息评分。

The blind detector scans all of 2024 without reading the 2024 schedule. It looks for the learned “dropoff first, pickup later” shape and treats the best anchor hour as predicted kickoff. Candidates are saved and hashed before ground truth is loaded.

## 严格口径 / Strict rules

- 训练：2022–2023。测试：2024。  
  Train on 2022–2023 and test on 2024.
- 2024 年 1–7 月只用于估计全年的整体流量比例，不使用比赛标签。  
  January–July 2024 is used only for an overall volume adjustment, without game labels.
- 伦敦比赛不属于 Soldier Field 主评分范围。  
  The London game is excluded from Soldier Field scoring.
- 2024 年 Soldier Field 季前赛单独标记，不计入常规赛主指标。  
  The 2024 Soldier Field preseason game is labeled separately.
- 本数据是完成订单，不是全部叫车需求、观众人数、取消订单或司机供给。  
  These are completed trips, not all ride requests, attendance, cancellations, or driver supply.

In [ ]:
from getpass import getpass
# Cell 2 — Imports and configuration / 导入包和配置

from pathlib import Path
from IPython.display import display
import hashlib
import json
import math
import os
import warnings

import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymysql
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (13, 6)
plt.rcParams["axes.unicode_minus"] = False

# Chinese font / 中文字体
font_candidates = [
    "PingFang SC",
    "Arial Unicode MS",
    "Heiti TC",
    "STHeiti",
    "Songti SC",
    "Noto Sans CJK SC",
]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
CHINESE_FONT = next(
    (font for font in font_candidates if font in available_fonts),
    None,
)
if CHINESE_FONT:
    plt.rcParams["font.sans-serif"] = [CHINESE_FONT, "DejaVu Sans"]

# Paths / 路径
PROJECT_DIR = Path(
    os.getenv("CHICAGO_TNP_PROJECT_DIR", str(Path.cwd()))
).expanduser().resolve()
OUTPUT_DIR = PROJECT_DIR / "notebook_outputs_bears_2024_blind_test"
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# MatrixOne / MatrixOne 配置
MO_HOST = os.getenv("MATRIXONE_HOST", "127.0.0.1")
MO_PORT = int(os.getenv("MATRIXONE_PORT", "6001"))
MO_USER = os.getenv("MATRIXONE_USER", "root")
MO_PASSWORD = os.getenv("MATRIXONE_PASSWORD") or getpass("MatrixOne password / MatrixOne 密码: ")
MO_DATABASE = os.getenv("MATRIXONE_DATABASE", "chicago_tnp")
ANALYSIS_TABLE = os.getenv(
    "MATRIXONE_ANALYSIS_TABLE", "unified_trips_h3_res9"
)

# Phase-1 core cells, fixed before looking at 2024. / 第一阶段已经确定的核心 H3，不使用 2024 重新选区。
CORE_H3_CELLS = [
    "892664c1b0bffff",
    "892664c1b47ffff",
]

TRAIN_START = pd.Timestamp("2022-01-01 00:00:00")
TRAIN_END = pd.Timestamp("2024-01-01 00:00:00")
TEST_START = pd.Timestamp("2024-01-01 00:00:00")
TEST_END = pd.Timestamp("2025-01-01 00:00:00")
CALIBRATION_END = pd.Timestamp("2024-08-01 00:00:00")

RELATIVE_HOURS = list(range(-6, 9))
ARRIVAL_HOURS = list(range(-4, 1))
EXIT_HOURS = [3, 4, 5]
TRAIN_EVENT_EXCLUSION_HOURS = list(range(-8, 11))

# Blind detection / 盲测参数
NEGATIVE_ANCHOR_STEP_HOURS = 4
BLIND_TOP_K = 25
MIN_CANDIDATE_SEPARATION_HOURS = 36
MATCH_TOLERANCE_HOURS = 2
RANDOM_STATE = 42

FORCE_REFRESH_HOURLY = False

print("Project directory / 项目目录:", PROJECT_DIR)
print("Output directory / 输出目录:", OUTPUT_DIR)
print("Analysis table / 分析表:", ANALYSIS_TABLE)
print("Fixed core H3 / 固定核心 H3:", CORE_H3_CELLS)

In [ ]:
# Cell 19 - MatrixOne helpers / MatrixOne 辅助函数

conn = pymysql.connect(
    host=MO_HOST,
    port=MO_PORT,
    user=MO_USER,
    password=MO_PASSWORD,
    database=MO_DATABASE,
    charset="utf8mb4",
    connect_timeout=30,
    read_timeout=7200,
    write_timeout=7200,
    autocommit=True,
)


def query_df(sql, params=None):
    """
    Run SQL and return a DataFrame.
    执行 SQL 并返回 DataFrame。
    """
    global conn
    try:
        conn.ping(reconnect=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return pd.read_sql_query(sql, conn, params=params)
    except Exception:
        try:
            conn.close()
        except Exception:
            pass
        conn = pymysql.connect(
            host=MO_HOST,
            port=MO_PORT,
            user=MO_USER,
            password=MO_PASSWORD,
            database=MO_DATABASE,
            charset="utf8mb4",
            connect_timeout=30,
            read_timeout=7200,
            write_timeout=7200,
            autocommit=True,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return pd.read_sql_query(sql, conn, params=params)


def sql_string(value):
    """
    Quote a Python value as a SQL string.
    把 Python 值转换成 SQL 字符串。
    """
    return "'" + str(value).replace("\\", "\\\\").replace("'", "''") + "'"


table_check = query_df(f"""
SELECT COUNT(*) AS n
FROM `{ANALYSIS_TABLE}`
LIMIT 1;
""")
print("Connected to MatrixOne / 已连接 MatrixOne")
display(table_check)

## B1. 固定训练赛程和空间范围
## 1. Freeze the training schedule and spatial scope

训练赛程只包含 2022–2023 年 Soldier Field 的 17 场常规赛主场比赛。核心 H3 来自第一阶段，进入本阶段后不再根据 2024 结果修改。

The training schedule contains the 17 regular-season Soldier Field home games from 2022–2023. Core H3 cells were fixed in Phase 1 and are not changed using 2024 results.

In [ ]:
# Cell 20 - Define 2022–2023 training games / 定义训练比赛

train_game_rows = [
    (2022, 1, "San Francisco 49ers", "2022-09-11 12:00:00"),
    (2022, 3, "Houston Texans", "2022-09-25 12:00:00"),
    (2022, 6, "Washington Commanders", "2022-10-13 19:15:00"),
    (2022, 9, "Miami Dolphins", "2022-11-06 12:00:00"),
    (2022, 10, "Detroit Lions", "2022-11-13 12:00:00"),
    (2022, 13, "Green Bay Packers", "2022-12-04 12:00:00"),
    (2022, 15, "Philadelphia Eagles", "2022-12-18 12:00:00"),
    (2022, 16, "Buffalo Bills", "2022-12-24 12:00:00"),
    (2022, 18, "Minnesota Vikings", "2023-01-08 12:00:00"),
    (2023, 1, "Green Bay Packers", "2023-09-10 15:25:00"),
    (2023, 4, "Denver Broncos", "2023-10-01 12:00:00"),
    (2023, 6, "Minnesota Vikings", "2023-10-15 12:00:00"),
    (2023, 7, "Las Vegas Raiders", "2023-10-22 12:00:00"),
    (2023, 10, "Carolina Panthers", "2023-11-09 19:15:00"),
    (2023, 14, "Detroit Lions", "2023-12-10 12:00:00"),
    (2023, 16, "Arizona Cardinals", "2023-12-24 15:25:00"),
    (2023, 17, "Atlanta Falcons", "2023-12-31 12:00:00"),
]

train_games = pd.DataFrame(
    train_game_rows,
    columns=["season", "week", "opponent", "scheduled_kickoff_local"],
)
train_games["scheduled_kickoff_local"] = pd.to_datetime(
    train_games["scheduled_kickoff_local"]
)
train_games["kickoff_hour"] = (
    train_games["scheduled_kickoff_local"].dt.floor("h")
)
train_games["game_id"] = train_games.apply(
    lambda row: (
        f"{row['season']}_W{int(row['week']):02d}_"
        f"{row['scheduled_kickoff_local']:%Y%m%d}"
    ),
    axis=1,
)
train_games["game_label"] = train_games.apply(
    lambda row: (
        f"{row['season']} W{int(row['week'])} vs {row['opponent']}"
    ),
    axis=1,
)

assert len(train_games) == 17
print("Training games / 训练比赛:", len(train_games))
display(train_games)

## B2. 从 MatrixOne 读取三年核心区小时客流
## 2. Query three years of core-zone hourly flow

查询只聚合两个固定核心 H3，不做大型空间 JOIN。pickup 使用行程开始时间，dropoff 使用行程结束时间。

The query aggregates only the two fixed core H3 cells. Pickup uses trip start time and dropoff uses trip end time.

In [ ]:
# Cell 21 - Query hourly pickup/dropoff / 查询小时 pickup/dropoff

core_sql = ", ".join(sql_string(cell) for cell in CORE_H3_CELLS)
pickup_cache = CACHE_DIR / "core_hourly_pickup_2022_2024.csv.gz"
dropoff_cache = CACHE_DIR / "core_hourly_dropoff_2022_2024.csv.gz"

if (
    pickup_cache.exists()
    and dropoff_cache.exists()
    and not FORCE_REFRESH_HOURLY
):
    pickup_hourly = pd.read_csv(
        pickup_cache, parse_dates=["hour_start"]
    )
    dropoff_hourly = pd.read_csv(
        dropoff_cache, parse_dates=["hour_start"]
    )
    print("Loaded hourly cache / 已读取小时缓存")
else:
    print("Query 1/2: pickup / 查询 1/2：pickup")
    pickup_hourly = query_df(f"""
    SELECT
        DATE_FORMAT(
            trip_start_timestamp, '%Y-%m-%d %H:00:00'
        ) AS hour_start,
        COUNT(*) AS pickup_count
    FROM `{ANALYSIS_TABLE}`
    WHERE pickup_h3 IN ({core_sql})
      AND trip_start_timestamp >= '2022-01-01'
      AND trip_start_timestamp < '2025-01-01'
      AND COALESCE(shared_trip_authorized, 0) = 0
    GROUP BY hour_start
    ORDER BY hour_start;
    """)

    print("Query 2/2: dropoff / 查询 2/2：dropoff")
    dropoff_hourly = query_df(f"""
    SELECT
        DATE_FORMAT(
            trip_end_timestamp, '%Y-%m-%d %H:00:00'
        ) AS hour_start,
        COUNT(*) AS dropoff_count
    FROM `{ANALYSIS_TABLE}`
    WHERE dropoff_h3 IN ({core_sql})
      AND trip_end_timestamp >= '2022-01-01'
      AND trip_end_timestamp < '2025-01-01'
      AND COALESCE(shared_trip_authorized, 0) = 0
    GROUP BY hour_start
    ORDER BY hour_start;
    """)

    pickup_hourly["hour_start"] = pd.to_datetime(
        pickup_hourly["hour_start"]
    )
    dropoff_hourly["hour_start"] = pd.to_datetime(
        dropoff_hourly["hour_start"]
    )
    pickup_hourly.to_csv(
        pickup_cache, index=False, compression="gzip"
    )
    dropoff_hourly.to_csv(
        dropoff_cache, index=False, compression="gzip"
    )

pickup_hourly["hour_start"] = pd.to_datetime(
    pickup_hourly["hour_start"]
)
dropoff_hourly["hour_start"] = pd.to_datetime(
    dropoff_hourly["hour_start"]
)
pickup_hourly["pickup_count"] = pd.to_numeric(
    pickup_hourly["pickup_count"], errors="coerce"
).fillna(0)
dropoff_hourly["dropoff_count"] = pd.to_numeric(
    dropoff_hourly["dropoff_count"], errors="coerce"
).fillna(0)

full_hours = pd.date_range(
    TRAIN_START,
    TEST_END - pd.Timedelta(hours=1),
    freq="h",
)
hourly = pd.DataFrame({"hour_start": full_hours})
hourly = (
    hourly
    .merge(pickup_hourly, on="hour_start", how="left")
    .merge(dropoff_hourly, on="hour_start", how="left")
)
hourly[["pickup_count", "dropoff_count"]] = hourly[
    ["pickup_count", "dropoff_count"]
].fillna(0)
hourly["year"] = hourly["hour_start"].dt.year
hourly["month"] = hourly["hour_start"].dt.month
hourly["weekday"] = hourly["hour_start"].dt.weekday
hourly["clock_hour"] = hourly["hour_start"].dt.hour

print("Hourly rows / 小时记录:", len(hourly))
print("Date range / 日期范围:", hourly["hour_start"].min(), "->", hourly["hour_start"].max())
display(hourly.groupby("year")[["pickup_count", "dropoff_count"]].sum())

## B3. 学习正常小时基线和 2024 流量比例
## 3. Learn the normal baseline and 2024 volume scale

正常基线使用非比赛窗口中的 2022–2023 数据，按“月份 × 星期几 × 钟点”计算中位数和 MAD。为了避免 MAD 很小时 z-score 无限放大，稳健尺度至少取 `sqrt(median+1)` 和 5。

The normal baseline uses non-game 2022–2023 hours grouped by month, weekday, and clock hour. A scale floor prevents unstable z-scores when MAD is very small.

2024 整体比例只使用 1–7 月实际客流与训练基线的中位数比值，不读取任何比赛标签。

The 2024 volume adjustment uses only the median actual-to-baseline ratio from January–July, without game labels.

In [ ]:
# Cell 22 - Build leakage-safe normal baseline / 构造无赛程泄漏的正常基线

excluded_train_hours = set()
for kickoff in train_games["kickoff_hour"]:
    for relative_hour in TRAIN_EVENT_EXCLUSION_HOURS:
        excluded_train_hours.add(
            kickoff + pd.Timedelta(hours=relative_hour)
        )

train_hourly = hourly[
    (hourly["hour_start"] >= TRAIN_START)
    & (hourly["hour_start"] < TRAIN_END)
].copy()
eligible_train = train_hourly[
    ~train_hourly["hour_start"].isin(excluded_train_hours)
].copy()


def median_absolute_deviation(values):
    """
    Return the median absolute deviation.
    返回中位数绝对偏差。
    """
    values = pd.Series(values).dropna().astype(float)
    median_value = values.median()
    return (values - median_value).abs().median()


profile_rows = []
for month in range(1, 13):
    for weekday in range(7):
        for clock_hour in range(24):
            group = eligible_train[
                (eligible_train["month"] == month)
                & (eligible_train["weekday"] == weekday)
                & (eligible_train["clock_hour"] == clock_hour)
            ]

            # Use adjacent months when the exact-month sample is small. / 当月样本过少时使用相邻月份。
            if len(group) < 8:
                month_distance = np.minimum(
                    (eligible_train["month"] - month).abs(),
                    12 - (eligible_train["month"] - month).abs(),
                )
                group = eligible_train[
                    (month_distance <= 1)
                    & (eligible_train["weekday"] == weekday)
                    & (
                        eligible_train["clock_hour"]
                        == clock_hour
                    )
                ]

            if group.empty:
                continue

            row = {
                "month": month,
                "weekday": weekday,
                "clock_hour": clock_hour,
                "control_n": len(group),
            }
            for flow in ["pickup", "dropoff"]:
                values = group[f"{flow}_count"].astype(float)
                median_value = values.median()
                mad_value = median_absolute_deviation(values)
                robust_scale = max(
                    1.4826 * mad_value,
                    math.sqrt(median_value + 1),
                    5.0,
                )
                row.update(
                    {
                        f"{flow}_normal_median": median_value,
                        f"{flow}_normal_p05": values.quantile(0.05),
                        f"{flow}_normal_p95": values.quantile(0.95),
                        f"{flow}_normal_scale": robust_scale,
                    }
                )
            profile_rows.append(row)

normal_profile = pd.DataFrame(profile_rows)
hourly_scored = hourly.merge(
    normal_profile,
    on=["month", "weekday", "clock_hour"],
    how="left",
    validate="many_to_one",
)

calibration = hourly_scored[
    (hourly_scored["hour_start"] >= TEST_START)
    & (hourly_scored["hour_start"] < CALIBRATION_END)
].copy()
volume_scale_2024 = {}
for flow in ["pickup", "dropoff"]:
    ratio = (
        calibration[f"{flow}_count"]
        / calibration[f"{flow}_normal_median"].replace(0, np.nan)
    )
    ratio = ratio.replace([np.inf, -np.inf], np.nan).dropna()
    ratio = ratio.clip(0.5, 2.0)
    volume_scale_2024[flow] = float(ratio.median())

for flow in ["pickup", "dropoff"]:
    is_2024 = hourly_scored["year"] == 2024
    hourly_scored[f"{flow}_expected"] = hourly_scored[
        f"{flow}_normal_median"
    ]
    hourly_scored.loc[
        is_2024, f"{flow}_expected"
    ] *= volume_scale_2024[flow]

    hourly_scored[f"{flow}_scale"] = hourly_scored[
        f"{flow}_normal_scale"
    ]
    hourly_scored.loc[
        is_2024, f"{flow}_scale"
    ] *= math.sqrt(volume_scale_2024[flow])

    hourly_scored[f"{flow}_z"] = (
        hourly_scored[f"{flow}_count"]
        - hourly_scored[f"{flow}_expected"]
    ) / hourly_scored[f"{flow}_scale"]

    hourly_scored[f"{flow}_log_excess"] = (
        np.log1p(hourly_scored[f"{flow}_count"])
        - np.log1p(hourly_scored[f"{flow}_expected"])
    )

hourly_scored = hourly_scored.sort_values(
    "hour_start"
).reset_index(drop=True)
hourly_lookup = hourly_scored.set_index("hour_start")

print("2024 volume scale / 2024 流量比例:", volume_scale_2024)
print(
    "Minimum training controls / 最少训练对照数:",
    int(normal_profile["control_n"].min()),
)
display(normal_profile.head())

## B4. 从 2022–2023 学习比赛事件模板
## 4. Learn the event template from 2022–2023

模板使用 `log(实际+1)-log(正常预期+1)`。0 表示接近普通日；正数表示高于普通日；负数表示低于普通日。中位数模板可以减少单场极端天气或节日比赛的影响。

The template uses `log(actual+1)-log(normal+1)`. Zero means normal, positive means higher than normal, and negative means lower. The median reduces the effect of unusual single games.

In [ ]:
# Cell 23 - Learn the 2022–2023 event template / 学习事件模板

template_rows = []
for game in train_games.itertuples(index=False):
    for relative_hour in RELATIVE_HOURS:
        target_hour = (
            game.kickoff_hour
            + pd.Timedelta(hours=relative_hour)
        )
        if target_hour not in hourly_lookup.index:
            continue
        source = hourly_lookup.loc[target_hour]
        template_rows.append(
            {
                "game_id": game.game_id,
                "game_label": game.game_label,
                "kickoff_hour": game.kickoff_hour,
                "relative_hour": relative_hour,
                "pickup_count": source["pickup_count"],
                "pickup_expected": source["pickup_expected"],
                "pickup_z": source["pickup_z"],
                "pickup_log_excess": source["pickup_log_excess"],
                "dropoff_count": source["dropoff_count"],
                "dropoff_expected": source["dropoff_expected"],
                "dropoff_z": source["dropoff_z"],
                "dropoff_log_excess": source["dropoff_log_excess"],
            }
        )

train_event_rows = pd.DataFrame(template_rows)
event_template = (
    train_event_rows
    .groupby("relative_hour", as_index=False)
    .agg(
        pickup_log_excess=(
            "pickup_log_excess", "median"
        ),
        dropoff_log_excess=(
            "dropoff_log_excess", "median"
        ),
        pickup_z=("pickup_z", "median"),
        dropoff_z=("dropoff_z", "median"),
    )
    .sort_values("relative_hour")
)
event_template.to_csv(
    OUTPUT_DIR / "training_event_template_2022_2023.csv",
    index=False,
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)
axes[0].plot(
    event_template["relative_hour"],
    event_template["pickup_log_excess"],
    marker="o",
    color="#1565c0",
    label="Pickup template",
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Pickup event template / pickup 事件模板")
axes[0].set_ylabel("Median log excess / 中位数 log excess")

axes[1].plot(
    event_template["relative_hour"],
    event_template["dropoff_log_excess"],
    marker="o",
    color="#c62828",
    label="Dropoff template",
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Dropoff event template / dropoff 事件模板")

for ax in axes:
    ax.axvline(0, color="black", linestyle=":", linewidth=2)
    ax.axvline(
        3.25,
        color="#2e7d32",
        linestyle="--",
        linewidth=1.5,
        label="Estimated end / 估算结束",
    )
    ax.set_xticks(RELATIVE_HOURS)
    ax.set_xlabel("Hours from kickoff / 距开球小时")
    ax.legend()

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "01_training_event_template.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()
display(event_template)

## B5. 已知 2024 比赛日和开球时间
# Test 1: known 2024 game dates and kickoff times

2024 Soldier Field 常规赛赛程来自 Chicago Bears 官方发布页面：

Official source:
https://www.chicagobears.com/news/2024-chicago-bears-regular-season-schedule-tickets-information-nfl-preseason-opponents-home-away-soldier-field

伦敦对 Jaguars 的比赛虽然是名义主场，但不在 Soldier Field，因此不纳入本测试。

The London Jaguars game was a designated home game but not played at Soldier Field, so it is excluded.

In [ ]:
# Cell 24 - Known 2024 Soldier Field schedule / 已知的 2024 Soldier Field 赛程

test1_schedule_rows = [
    (1, "Tennessee Titans", "2024-09-08 12:00:00"),
    (4, "Los Angeles Rams", "2024-09-29 12:00:00"),
    (5, "Carolina Panthers", "2024-10-06 12:00:00"),
    (10, "New England Patriots", "2024-11-10 12:00:00"),
    (11, "Green Bay Packers", "2024-11-17 12:00:00"),
    (12, "Minnesota Vikings", "2024-11-24 12:00:00"),
    (16, "Detroit Lions", "2024-12-22 12:00:00"),
    (17, "Seattle Seahawks", "2024-12-26 19:15:00"),
]

test1_games_2024 = pd.DataFrame(
    test1_schedule_rows,
    columns=["week", "opponent", "scheduled_kickoff_local"],
)
test1_games_2024["scheduled_kickoff_local"] = pd.to_datetime(
    test1_games_2024["scheduled_kickoff_local"]
)
test1_games_2024["kickoff_hour"] = (
    test1_games_2024["scheduled_kickoff_local"].dt.floor("h")
)
test1_games_2024["game_id"] = test1_games_2024.apply(
    lambda row: (
        f"2024_W{int(row['week']):02d}_"
        f"{row['scheduled_kickoff_local']:%Y%m%d}"
    ),
    axis=1,
)
test1_games_2024["game_label"] = test1_games_2024.apply(
    lambda row: f"2024 W{int(row['week'])} vs {row['opponent']}",
    axis=1,
)

assert len(test1_games_2024) == 8
display(test1_games_2024)

In [ ]:
# Cell 25 - Predict every event-relative hour / 预测每个相对小时客流

template_lookup = event_template.set_index("relative_hour")
test1_rows = []

for game in test1_games_2024.itertuples(index=False):
    for relative_hour in RELATIVE_HOURS:
        target_hour = (
            game.kickoff_hour
            + pd.Timedelta(hours=relative_hour)
        )
        source = hourly_lookup.loc[target_hour]
        template = template_lookup.loc[relative_hour]

        row = {
            "game_id": game.game_id,
            "game_label": game.game_label,
            "scheduled_kickoff_local": game.scheduled_kickoff_local,
            "kickoff_hour": game.kickoff_hour,
            "relative_hour": relative_hour,
            "event_hour_start": target_hour,
        }
        for flow in ["pickup", "dropoff"]:
            normal = float(source[f"{flow}_expected"])
            template_excess = float(
                template[f"{flow}_log_excess"]
            )
            prediction = max(
                0.0,
                math.expm1(
                    math.log1p(normal) + template_excess
                ),
            )
            actual = float(source[f"{flow}_count"])
            row.update(
                {
                    f"{flow}_actual": actual,
                    f"{flow}_normal": normal,
                    f"{flow}_predicted": prediction,
                    f"{flow}_error": actual - prediction,
                    f"{flow}_absolute_error": abs(
                        actual - prediction
                    ),
                    f"{flow}_pct_error": (
                        100 * (actual - prediction) / prediction
                        if prediction > 0 else np.nan
                    ),
                }
            )
        test1_rows.append(row)

test1_predictions = pd.DataFrame(test1_rows)
test1_predictions.to_csv(
    OUTPUT_DIR / "test1_known_schedule_hourly_predictions.csv",
    index=False,
)

metric_rows = []
for flow in ["pickup", "dropoff"]:
    actual = test1_predictions[f"{flow}_actual"].to_numpy()
    predicted = test1_predictions[
        f"{flow}_predicted"
    ].to_numpy()
    normal = test1_predictions[f"{flow}_normal"].to_numpy()
    metric_rows.append(
        {
            "flow": flow,
            "event_model_MAE": mean_absolute_error(
                actual, predicted
            ),
            "normal_baseline_MAE": mean_absolute_error(
                actual, normal
            ),
            "event_model_RMSE": math.sqrt(
                mean_squared_error(actual, predicted)
            ),
            "WAPE_pct": (
                100 * np.abs(actual - predicted).sum()
                / np.abs(actual).sum()
            ),
            "MAE_improvement_vs_normal_pct": (
                100
                * (
                    mean_absolute_error(actual, normal)
                    - mean_absolute_error(actual, predicted)
                )
                / mean_absolute_error(actual, normal)
            ),
        }
    )

test1_metrics = pd.DataFrame(metric_rows)
test1_metrics.to_csv(
    OUTPUT_DIR / "test1_known_schedule_metrics.csv",
    index=False,
)
display(test1_metrics)
display(test1_predictions.head(15))

In [ ]:
# Cell 26 - Plot known-schedule average prediction / 绘制已知赛程的平均预测

test1_average = (
    test1_predictions
    .groupby("relative_hour", as_index=False)
    .agg(
        pickup_actual=("pickup_actual", "mean"),
        pickup_predicted=("pickup_predicted", "mean"),
        pickup_normal=("pickup_normal", "mean"),
        dropoff_actual=("dropoff_actual", "mean"),
        dropoff_predicted=("dropoff_predicted", "mean"),
        dropoff_normal=("dropoff_normal", "mean"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)
for ax, flow, color in [
    (axes[0], "pickup", "#1565c0"),
    (axes[1], "dropoff", "#c62828"),
]:
    ax.plot(
        test1_average["relative_hour"],
        test1_average[f"{flow}_actual"],
        marker="o",
        color=color,
        linewidth=2.5,
        label="Actual 2024 / 2024 实际",
    )
    ax.plot(
        test1_average["relative_hour"],
        test1_average[f"{flow}_predicted"],
        marker="o",
        linestyle="--",
        color="#2e7d32",
        linewidth=2.2,
        label="Event prediction / 事件预测",
    )
    ax.plot(
        test1_average["relative_hour"],
        test1_average[f"{flow}_normal"],
        linestyle=":",
        color="#616161",
        linewidth=2,
        label="Normal baseline / 普通日基线",
    )
    ax.axvline(0, color="black", linestyle=":", linewidth=2)
    ax.axvline(
        3.25, color="#7b1fa2", linestyle="--", linewidth=1.5
    )
    ax.set_xticks(RELATIVE_HOURS)
    ax.set_xlabel("Hours from kickoff / 距开球小时")
    ax.set_ylabel("Trips per hour / 每小时订单")
    ax.set_title(f"Known-schedule {flow} / 已知赛程 {flow}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "02_test1_average_prediction.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Cell 27 - Plot each known 2024 game / 绘制每场已知比赛

n_games = len(test1_games_2024)
fig, axes = plt.subplots(
    n_games,
    2,
    figsize=(18, 3.5 * n_games),
    sharex=True,
)

for row_idx, game in enumerate(
    test1_games_2024.itertuples(index=False)
):
    data = test1_predictions[
        test1_predictions["game_id"] == game.game_id
    ].sort_values("relative_hour")

    for col_idx, flow, color in [
        (0, "pickup", "#1565c0"),
        (1, "dropoff", "#c62828"),
    ]:
        ax = axes[row_idx, col_idx]
        ax.plot(
            data["relative_hour"],
            data[f"{flow}_actual"],
            marker="o",
            color=color,
            label="Actual / 实际",
        )
        ax.plot(
            data["relative_hour"],
            data[f"{flow}_predicted"],
            marker="o",
            linestyle="--",
            color="#2e7d32",
            label="Predicted / 预测",
        )
        ax.plot(
            data["relative_hour"],
            data[f"{flow}_normal"],
            linestyle=":",
            color="#757575",
            label="Normal / 普通日",
        )
        ax.axvline(0, color="black", linestyle=":", linewidth=1.8)
        ax.axvline(
            3.25,
            color="#7b1fa2",
            linestyle="--",
            linewidth=1.3,
        )
        ax.set_title(f"{game.game_label} — {flow}")
        ax.set_xticks(RELATIVE_HOURS)
        ax.set_xlabel("Hours from kickoff / 距开球小时")
        ax.set_ylabel("Trips per hour / 每小时订单")
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "03_test1_each_game_prediction.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## B6. 不提供 2024 比赛日的盲测
# Test 2: blind detection without 2024 game dates

盲测模型只使用：

The blind model uses only:

1. 2022–2023 正常基线；  
   the 2022–2023 normal baseline;
2. 2022–2023 的 17 个已知比赛锚点；  
   the 17 known 2022–2023 game anchors;
3. 2022–2023 非比赛小时作为负样本；  
   non-game 2022–2023 anchors as negatives;
4. 2024 pickup/dropoff 时序本身。  
   the 2024 pickup/dropoff series itself.

测试 1 的 2024 赛程变量在下面会被删除。候选生成函数不接收任何 2024 赛程参数。

The Test-1 schedule variable is deleted below. Candidate generation receives no 2024 schedule input.

In [ ]:
# Cell 28 - Build blind-detector features / 构造盲测特征

DETECTOR_FEATURES = [
    "arrival_dropoff_mean_z",
    "arrival_dropoff_peak_z",
    "exit_pickup_mean_z",
    "exit_pickup_peak_z",
    "template_cosine",
    "template_correlation",
    "peak_order_valid",
]

template_vector = np.concatenate(
    [
        event_template["pickup_log_excess"].to_numpy(),
        event_template["dropoff_log_excess"].to_numpy(),
    ]
)


def safe_cosine(left, right):
    """
    Calculate cosine similarity safely.
    安全计算余弦相似度。
    """
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    if denominator == 0:
        return 0.0
    return float(np.dot(left, right) / denominator)


def safe_correlation(left, right):
    """
    Calculate correlation safely.
    安全计算相关系数。
    """
    if np.std(left) == 0 or np.std(right) == 0:
        return 0.0
    return float(np.corrcoef(left, right)[0, 1])


def extract_anchor_features(anchor_hour):
    """
    Build event-shape features around one proposed kickoff hour.
    围绕一个候选开球小时构造事件形状特征。
    """
    anchor_hour = pd.Timestamp(anchor_hour)
    target_hours = [
        anchor_hour + pd.Timedelta(hours=relative_hour)
        for relative_hour in RELATIVE_HOURS
    ]
    if any(hour not in hourly_lookup.index for hour in target_hours):
        return None

    window = hourly_lookup.loc[target_hours].copy()
    window["relative_hour"] = RELATIVE_HOURS
    candidate_vector = np.concatenate(
        [
            window["pickup_log_excess"].to_numpy(),
            window["dropoff_log_excess"].to_numpy(),
        ]
    )

    arrival = window[
        window["relative_hour"].isin(ARRIVAL_HOURS)
    ]
    exit_window = window[
        window["relative_hour"].isin(EXIT_HOURS)
    ]
    dropoff_peak_relative_hour = int(
        arrival.loc[arrival["dropoff_z"].idxmax(), "relative_hour"]
    )
    pickup_peak_relative_hour = int(
        exit_window.loc[
            exit_window["pickup_z"].idxmax(), "relative_hour"
        ]
    )

    return {
        "anchor_hour": anchor_hour,
        "arrival_dropoff_mean_z": float(
            arrival["dropoff_z"].mean()
        ),
        "arrival_dropoff_peak_z": float(
            arrival["dropoff_z"].max()
        ),
        "exit_pickup_mean_z": float(
            exit_window["pickup_z"].mean()
        ),
        "exit_pickup_peak_z": float(
            exit_window["pickup_z"].max()
        ),
        "template_cosine": safe_cosine(
            candidate_vector, template_vector
        ),
        "template_correlation": safe_correlation(
            candidate_vector, template_vector
        ),
        "dropoff_peak_relative_hour": dropoff_peak_relative_hour,
        "pickup_peak_relative_hour": pickup_peak_relative_hour,
        "peak_order_valid": float(
            dropoff_peak_relative_hour <= 0
            and pickup_peak_relative_hour >= 3
        ),
    }


# Save Test 1 output, then hide the schedule before blind scanning. / 保存测试一结果，盲扫前隐藏赛程。
test1_schedule_for_reporting = test1_games_2024[
    ["game_id", "game_label"]
].copy()
del test1_games_2024

print(
    "2024 schedule removed before blind generation / "
    "生成盲测候选前已删除 2024 赛程变量:",
    "test1_games_2024" not in globals(),
)

In [ ]:
# Cell 29 - Train detector on 2022–2023 only / 只使用 2022–2023 训练检测器

positive_rows = []
for game in train_games.itertuples(index=False):
    features = extract_anchor_features(game.kickoff_hour)
    if features is not None:
        features.update(
            {
                "label": 1,
                "sample_type": "real_game",
                "sample_id": game.game_id,
            }
        )
        positive_rows.append(features)

excluded_negative_hours = set()
for kickoff in train_games["kickoff_hour"]:
    for offset in range(-18, 19):
        excluded_negative_hours.add(
            kickoff + pd.Timedelta(hours=offset)
        )

negative_rows = []
negative_anchors = pd.date_range(
    pd.Timestamp("2022-07-01 06:00:00"),
    pd.Timestamp("2023-12-31 15:00:00"),
    freq=f"{NEGATIVE_ANCHOR_STEP_HOURS}h",
)
for anchor in negative_anchors:
    if anchor in excluded_negative_hours:
        continue
    features = extract_anchor_features(anchor)
    if features is None:
        continue
    features.update(
        {
            "label": 0,
            "sample_type": "non_game",
            "sample_id": f"negative_{anchor:%Y%m%d_%H}",
        }
    )
    negative_rows.append(features)

detector_training = pd.DataFrame(
    positive_rows + negative_rows
)
X_train_detector = detector_training[DETECTOR_FEATURES]
y_train_detector = detector_training["label"]

detector_model = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                C=0.5,
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
detector_model.fit(X_train_detector, y_train_detector)
detector_training["event_probability"] = detector_model.predict_proba(
    X_train_detector
)[:, 1]

train_auc = roc_auc_score(
    y_train_detector,
    detector_training["event_probability"],
)
coefficients = pd.DataFrame(
    {
        "feature": DETECTOR_FEATURES,
        "standardized_coefficient": detector_model.named_steps[
            "model"
        ].coef_[0],
    }
).sort_values(
    "standardized_coefficient", ascending=False
)

print("Training positives / 训练正样本:", len(positive_rows))
print("Training negatives / 训练负样本:", len(negative_rows))
print(
    "Training AUC is descriptive only / 训练 AUC 只作描述:",
    round(train_auc, 4),
)
display(coefficients)
display(
    detector_training.groupby("sample_type")[
        "event_probability"
    ].describe()
)

In [ ]:
# Cell 30 - Scan all 2024 anchors without schedule / 不使用赛程扫描 2024 全部候选小时

blind_rows = []
blind_anchors = pd.date_range(
    TEST_START + pd.Timedelta(hours=6),
    TEST_END - pd.Timedelta(hours=9),
    freq="h",
)
for index, anchor in enumerate(blind_anchors, start=1):
    features = extract_anchor_features(anchor)
    if features is not None:
        blind_rows.append(features)
    if index % 2000 == 0:
        print(
            f"Scanned {index:,}/{len(blind_anchors):,} anchors / "
            f"已扫描 {index:,}/{len(blind_anchors):,}"
        )

blind_all_scores = pd.DataFrame(blind_rows)
blind_all_scores["event_probability"] = detector_model.predict_proba(
    blind_all_scores[DETECTOR_FEATURES]
)[:, 1]
blind_all_scores = blind_all_scores.sort_values(
    "event_probability", ascending=False
).reset_index(drop=True)


def select_separated_candidates(
    score_frame,
    top_k=BLIND_TOP_K,
    separation_hours=MIN_CANDIDATE_SEPARATION_HOURS,
):
    """
    Keep high-scoring anchors separated in time.
    保留时间上彼此分开的高分候选。
    """
    selected = []
    for row in score_frame.itertuples(index=False):
        if any(
            abs(
                (
                    row.anchor_hour - item["anchor_hour"]
                ).total_seconds()
            )
            < separation_hours * 3600
            for item in selected
        ):
            continue
        selected.append(row._asdict())
        if len(selected) >= top_k:
            break
    return pd.DataFrame(selected)


blind_candidates = select_separated_candidates(blind_all_scores)
blind_candidates.insert(
    0,
    "blind_rank",
    np.arange(1, len(blind_candidates) + 1),
)
blind_candidates_path = (
    OUTPUT_DIR / "test2_blind_candidates_before_truth.csv"
)
blind_candidates.to_csv(blind_candidates_path, index=False)

candidate_bytes = blind_candidates.to_csv(index=False).encode("utf-8")
BLIND_CANDIDATE_SHA256 = hashlib.sha256(
    candidate_bytes
).hexdigest()
(
    OUTPUT_DIR / "test2_blind_candidates_sha256.txt"
).write_text(BLIND_CANDIDATE_SHA256 + "\n", encoding="utf-8")

print("Blind candidates frozen / 盲测候选已固定:", len(blind_candidates))
print("SHA-256:", BLIND_CANDIDATE_SHA256)
display(blind_candidates)

In [ ]:
# Cell 31 - Plot blind score timeline and top candidate signatures / 绘制盲测分数时间线和候选形状

blind_timeline = blind_all_scores.sort_values("anchor_hour")
fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(
    blind_timeline["anchor_hour"],
    blind_timeline["event_probability"],
    color="#546e7a",
    linewidth=0.8,
    alpha=0.8,
)
ax.scatter(
    blind_candidates["anchor_hour"],
    blind_candidates["event_probability"],
    color="#c62828",
    s=32,
    label="Frozen blind candidates / 固定盲测候选",
    zorder=3,
)
ax.set_title(
    "Blind event probability in 2024 — no schedule used / "
    "2024 盲测事件概率（未使用赛程）"
)
ax.set_xlabel("Candidate kickoff hour / 候选开球小时")
ax.set_ylabel("Event probability / 事件概率")
ax.legend()
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "04_test2_blind_score_timeline.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

top_plot_candidates = blind_candidates.head(12)
fig, axes = plt.subplots(
    4, 3, figsize=(18, 15), sharex=True
)
axes = axes.ravel()
for ax, candidate in zip(
    axes, top_plot_candidates.itertuples(index=False)
):
    target_hours = [
        candidate.anchor_hour + pd.Timedelta(hours=relative_hour)
        for relative_hour in RELATIVE_HOURS
    ]
    window = hourly_lookup.loc[target_hours].copy()
    ax.plot(
        RELATIVE_HOURS,
        window["dropoff_z"],
        marker="o",
        color="#c62828",
        label="Dropoff z",
    )
    ax.plot(
        RELATIVE_HOURS,
        window["pickup_z"],
        marker="o",
        color="#1565c0",
        label="Pickup z",
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linestyle=":", linewidth=1.5)
    ax.set_title(
        f"#{candidate.blind_rank} {candidate.anchor_hour:%Y-%m-%d %H:%M}\n"
        f"p={candidate.event_probability:.3f}"
    )
    ax.set_xticks(RELATIVE_HOURS)
    ax.set_xlabel("Hours from candidate / 距候选小时")
    ax.set_ylabel("Stable z-score")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "05_test2_top_candidate_signatures.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## B7. 候选固定后加载真实赛程评分
## 5. Load ground truth only after candidates are frozen

下面重新创建 2024 真实赛程。它只用于评分，不参与候选生成。主评分包含 8 场 Soldier Field 常规赛。

Ground truth is recreated below only for scoring. The primary score contains eight Soldier Field regular-season games.

另列一场 2024 年 8 月 17 日 Soldier Field 季前赛。如果盲测找到它，应标记为真实 Bears 活动，而不是普通假阳性。

The August 17 Soldier Field preseason game is labeled separately. A matching candidate is a real Bears event, not an ordinary false positive.

In [ ]:
# Cell 32 - Load 2024 ground truth after freezing candidates / 候选固定后加载 2024 真实赛程

ground_truth_rows = [
    (1, "Tennessee Titans", "2024-09-08 12:00:00"),
    (4, "Los Angeles Rams", "2024-09-29 12:00:00"),
    (5, "Carolina Panthers", "2024-10-06 12:00:00"),
    (10, "New England Patriots", "2024-11-10 12:00:00"),
    (11, "Green Bay Packers", "2024-11-17 12:00:00"),
    (12, "Minnesota Vikings", "2024-11-24 12:00:00"),
    (16, "Detroit Lions", "2024-12-22 12:00:00"),
    (17, "Seattle Seahawks", "2024-12-26 19:15:00"),
]
ground_truth_games_2024 = pd.DataFrame(
    ground_truth_rows,
    columns=["week", "opponent", "scheduled_kickoff_local"],
)
ground_truth_games_2024["scheduled_kickoff_local"] = pd.to_datetime(
    ground_truth_games_2024["scheduled_kickoff_local"]
)
ground_truth_games_2024["kickoff_hour"] = (
    ground_truth_games_2024[
        "scheduled_kickoff_local"
    ].dt.floor("h")
)
ground_truth_games_2024["game_label"] = ground_truth_games_2024.apply(
    lambda row: f"2024 W{int(row['week'])} vs {row['opponent']}",
    axis=1,
)

known_secondary_events_2024 = pd.DataFrame(
    [
        {
            "event_label": "2024 preseason vs Cincinnati Bengals",
            "event_time": pd.Timestamp("2024-08-17 12:00:00"),
            "event_type": "Bears preseason at Soldier Field",
        },
        {
            "event_label": "2024 vs Jacksonville Jaguars",
            "event_time": pd.Timestamp("2024-10-13 08:30:00"),
            "event_type": "Designated home game in London — not Soldier Field",
        },
    ]
)

print("Frozen candidate SHA-256 / 固定候选 SHA-256:")
print(BLIND_CANDIDATE_SHA256)
display(ground_truth_games_2024)
display(known_secondary_events_2024)

In [ ]:
# Cell 33 - Match blind candidates to real games / 盲测候选匹配真实比赛

comparison_rows = []
for game in ground_truth_games_2024.itertuples(index=False):
    candidate_distances = (
        blind_candidates["anchor_hour"] - game.kickoff_hour
    ).abs()
    nearest_index = candidate_distances.idxmin()
    nearest = blind_candidates.loc[nearest_index]
    signed_error_hours = (
        nearest["anchor_hour"] - game.kickoff_hour
    ).total_seconds() / 3600
    comparison_rows.append(
        {
            "game_label": game.game_label,
            "actual_kickoff": game.scheduled_kickoff_local,
            "actual_kickoff_hour": game.kickoff_hour,
            "predicted_anchor": nearest["anchor_hour"],
            "blind_rank": int(nearest["blind_rank"]),
            "event_probability": nearest["event_probability"],
            "signed_time_error_hours": signed_error_hours,
            "absolute_time_error_hours": abs(signed_error_hours),
            "same_calendar_date": (
                nearest["anchor_hour"].date()
                == game.kickoff_hour.date()
            ),
            "detected_within_1h": abs(signed_error_hours) <= 1,
            "detected_within_2h": (
                abs(signed_error_hours) <= MATCH_TOLERANCE_HOURS
            ),
        }
    )

blind_game_comparison = pd.DataFrame(comparison_rows)
blind_game_comparison.to_csv(
    OUTPUT_DIR / "test2_blind_vs_true_games.csv",
    index=False,
)

secondary_rows = []
for event in known_secondary_events_2024.itertuples(index=False):
    distances = (
        blind_candidates["anchor_hour"] - event.event_time
    ).abs()
    nearest_index = distances.idxmin()
    nearest = blind_candidates.loc[nearest_index]
    error_hours = (
        nearest["anchor_hour"] - event.event_time
    ).total_seconds() / 3600
    secondary_rows.append(
        {
            "event_label": event.event_label,
            "event_type": event.event_type,
            "event_time": event.event_time,
            "nearest_candidate": nearest["anchor_hour"],
            "blind_rank": int(nearest["blind_rank"]),
            "absolute_time_error_hours": abs(error_hours),
            "within_2h": abs(error_hours) <= 2,
        }
    )
secondary_event_comparison = pd.DataFrame(secondary_rows)

print(
    "Regular-season games detected within ±1h / "
    "常规赛在 ±1 小时内识别:",
    int(blind_game_comparison["detected_within_1h"].sum()),
    "/",
    len(blind_game_comparison),
)
print(
    "Regular-season games detected within ±2h / "
    "常规赛在 ±2 小时内识别:",
    int(blind_game_comparison["detected_within_2h"].sum()),
    "/",
    len(blind_game_comparison),
)
print(
    "Correct calendar date / 日期识别正确:",
    int(blind_game_comparison["same_calendar_date"].sum()),
    "/",
    len(blind_game_comparison),
)
display(blind_game_comparison)
display(secondary_event_comparison)

In [ ]:
# Cell 34 - Top-K blind detection metrics / Top-K 盲测指标

top_k_rows = []
for top_k in [8, 12, 20, 25]:
    candidates_k = blind_candidates.head(top_k)
    matched_truth = set()
    matched_candidates = set()

    candidate_pairs = []
    for candidate in candidates_k.itertuples(index=False):
        for game_index, game in ground_truth_games_2024.iterrows():
            distance_hours = abs(
                (
                    candidate.anchor_hour - game["kickoff_hour"]
                ).total_seconds()
            ) / 3600
            if distance_hours <= MATCH_TOLERANCE_HOURS:
                candidate_pairs.append(
                    (
                        distance_hours,
                        int(candidate.blind_rank),
                        game_index,
                    )
                )

    for _, candidate_rank, game_index in sorted(candidate_pairs):
        if candidate_rank in matched_candidates:
            continue
        if game_index in matched_truth:
            continue
        matched_candidates.add(candidate_rank)
        matched_truth.add(game_index)

    top_k_rows.append(
        {
            "top_k": top_k,
            "matched_regular_season_games": len(matched_truth),
            "total_regular_season_games": len(
                ground_truth_games_2024
            ),
            "recall_within_2h": (
                len(matched_truth) / len(ground_truth_games_2024)
            ),
            "precision_within_2h": (
                len(matched_candidates) / len(candidates_k)
            ),
        }
    )

blind_top_k_metrics = pd.DataFrame(top_k_rows)
blind_top_k_metrics.to_csv(
    OUTPUT_DIR / "test2_blind_top_k_metrics.csv",
    index=False,
)
display(blind_top_k_metrics)

evaluated_candidates = blind_candidates.copy()
evaluated_candidates["nearest_regular_game"] = None
evaluated_candidates["hours_to_nearest_regular_game"] = np.nan
evaluated_candidates["match_type"] = "unmatched_candidate"

for index, candidate in evaluated_candidates.iterrows():
    distances = (
        ground_truth_games_2024["kickoff_hour"]
        - candidate["anchor_hour"]
    ).abs()
    nearest_idx = distances.idxmin()
    nearest_game = ground_truth_games_2024.loc[nearest_idx]
    hours = (
        distances.loc[nearest_idx].total_seconds() / 3600
    )
    evaluated_candidates.loc[
        index, "nearest_regular_game"
    ] = nearest_game["game_label"]
    evaluated_candidates.loc[
        index, "hours_to_nearest_regular_game"
    ] = hours
    if hours <= MATCH_TOLERANCE_HOURS:
        evaluated_candidates.loc[
            index, "match_type"
        ] = "regular_season_match"

preseason_time = pd.Timestamp("2024-08-17 12:00:00")
preseason_distance = (
    evaluated_candidates["anchor_hour"] - preseason_time
).abs().dt.total_seconds() / 3600
evaluated_candidates.loc[
    preseason_distance <= MATCH_TOLERANCE_HOURS,
    "match_type",
] = "preseason_match"

evaluated_candidates.to_csv(
    OUTPUT_DIR / "test2_blind_candidates_evaluated.csv",
    index=False,
)
display(
    evaluated_candidates[
        [
            "blind_rank",
            "anchor_hour",
            "event_probability",
            "match_type",
            "nearest_regular_game",
            "hours_to_nearest_regular_game",
        ]
    ]
)

In [ ]:
# Cell 35 - Plot blind candidates against truth / 绘制盲测与真实赛程对比

fig, ax = plt.subplots(figsize=(18, 6))
timeline = blind_all_scores.sort_values("anchor_hour")
ax.plot(
    timeline["anchor_hour"],
    timeline["event_probability"],
    color="#90a4ae",
    linewidth=0.7,
    label="Blind event score / 盲测事件分数",
)
ax.scatter(
    blind_candidates["anchor_hour"],
    blind_candidates["event_probability"],
    color="#c62828",
    s=32,
    label="Frozen blind candidates / 固定盲测候选",
    zorder=3,
)
for index, game in ground_truth_games_2024.iterrows():
    ax.axvline(
        game["kickoff_hour"],
        color="#2e7d32",
        linewidth=1.5,
        alpha=0.75,
        label="True Soldier Field game / 真实比赛"
        if index == 0 else None,
    )
ax.axvline(
    pd.Timestamp("2024-08-17 12:00:00"),
    color="#f9a825",
    linestyle="--",
    linewidth=1.8,
    label="Known preseason game / 已知季前赛",
)
ax.set_title(
    "Blind candidates revealed against 2024 truth / "
    "固定盲测候选与 2024 真实赛程"
)
ax.set_xlabel("Time / 时间")
ax.set_ylabel("Event probability / 事件概率")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "06_test2_blind_candidates_vs_truth.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
comparison_plot = blind_game_comparison.sort_values(
    "actual_kickoff"
)
colors = np.where(
    comparison_plot["detected_within_2h"],
    "#2e7d32",
    "#c62828",
)
ax.barh(
    comparison_plot["game_label"],
    comparison_plot["signed_time_error_hours"],
    color=colors,
)
ax.axvline(0, color="black", linewidth=1)
ax.axvline(
    -MATCH_TOLERANCE_HOURS,
    color="#7b1fa2",
    linestyle="--",
)
ax.axvline(
    MATCH_TOLERANCE_HOURS,
    color="#7b1fa2",
    linestyle="--",
)
ax.set_title(
    "Predicted kickoff-time error / 预测开球时间误差"
)
ax.set_xlabel(
    "Predicted minus actual hours / 预测减实际（小时）"
)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "07_test2_kickoff_time_error.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## B8. 结果解释规则
## 6. Interpretation rules

- 测试 1 的事件预测优于普通日基线，表示 2022–2023 事件模板对 2024 有迁移价值。  
  If the event prediction beats the normal baseline, the historical event template transfers to 2024.
- 测试 2 的 `±1h` 表示预测自然小时与真实开球自然小时最多相差一小时；`±2h` 是较宽松的事件识别标准。  
  `±1h` measures kickoff-hour accuracy; `±2h` is a looser event-detection standard.
- 未匹配候选不应立刻称为错误。它可能对应 Soldier Field 的其他比赛、演唱会或附近大型活动。  
  An unmatched candidate may be another Soldier Field or nearby event, not necessarily noise.
- 训练只有 17 个正样本，盲测结果是事件检测 proof of concept，不是生产系统。  
  With only 17 positive training games, this is a proof of concept rather than a production detector.

In [ ]:
# Cell 36 - Final validation and output list / 最终验证和输出清单

checks = pd.DataFrame(
    [
        {
            "check": "training_games",
            "value": len(train_games),
            "expected": 17,
            "passed": len(train_games) == 17,
        },
        {
            "check": "test1_predictions",
            "value": len(test1_predictions),
            "expected": 8 * len(RELATIVE_HOURS),
            "passed": (
                len(test1_predictions)
                == 8 * len(RELATIVE_HOURS)
            ),
        },
        {
            "check": "blind_candidates",
            "value": len(blind_candidates),
            "expected": BLIND_TOP_K,
            "passed": len(blind_candidates) == BLIND_TOP_K,
        },
        {
            "check": "blind_candidate_hash",
            "value": len(BLIND_CANDIDATE_SHA256),
            "expected": 64,
            "passed": len(BLIND_CANDIDATE_SHA256) == 64,
        },
        {
            "check": "ground_truth_games",
            "value": len(ground_truth_games_2024),
            "expected": 8,
            "passed": len(ground_truth_games_2024) == 8,
        },
        {
            "check": "minimum_normal_controls",
            "value": int(normal_profile["control_n"].min()),
            "expected": ">= 8",
            "passed": normal_profile["control_n"].min() >= 8,
        },
    ]
)
checks.to_csv(
    OUTPUT_DIR / "run_validation_checks.csv", index=False
)
display(checks)

print("\nKey Test-1 metrics / 测试1关键指标:")
display(test1_metrics)
print("\nKey Test-2 comparison / 测试2逐场结果:")
display(blind_game_comparison)
print("\nTop-K metrics / Top-K 指标:")
display(blind_top_k_metrics)

print("\nOutput directory / 输出目录:", OUTPUT_DIR)
print("\nGenerated files / 已生成文件:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print("-", path.name)

if checks["passed"].all():
    print("\nAll structural checks passed. / 所有结构检查通过。")
else:
    print(
        "\nSome structural checks require review. / "
        "部分结构检查需要确认。"
    )

## Result snapshot / 结果快照

- The event template is learned from 17 home games in 2022-2023 and checked on 8 Soldier Field games in 2024. / 赛事模板从 2022-2023 的 17 场主场比赛中学习，并在 2024 的 8 场 Soldier Field 比赛上检查。
- With the 2024 schedule supplied, pickup MAE improves by 44.4% and dropoff MAE by 68.5% against the normal-day baseline. / 提供 2024 赛程后，相比普通日基线，pickup MAE 改善 44.4%，dropoff MAE 改善 68.5%。
- Without the schedule, Top 8 candidates match 5 of 8 games; Top 25 candidates cover all 8 games with lower precision. / 不提供赛程时，Top 8 候选匹配 5 场；Top 25 候选覆盖全部 8 场，但精确率较低。

**Next / 下一步:** Notebook 08 turns the measured event pattern into repeated business scenarios.